In [9]:
import os
import glob
import itertools
import h5py
import numpy as np
import matplotlib.pyplot as plt
import gvar as gv
import lsqfit

#hadron, kinematics definition
hadron = "pion"              #"pion" or "proton"
frame = "forward"            #"forward" or "sym"
q = (0,0,0)                  #the q of a cell that does not average over q;  forward: (0,0,0);  sym: (2,0,0), (-2,0,0), (0,2,0) or (0,-2,0)
project = True               #proton only: True = positive-parity files, False = open Dirac files; For now, no opec dirac proton data, to be done later
ncfg = 250
Lt = 96
Ls = 32
hyp = 8                      #hyp steps to be used 

#Here a tricky thing which turns out to be harmless
#<3pt>=<2ptFF>-<2pt><FF>, but for off-forward case, where does e^-iqxsrc go in the vacuum term.
#For "2pt", it's <2pt*e^-iqxsrc><FF>, by translation invariance, the first term is extremely close to zero but not zero
#For "FF",  it's <2pt><FF*e^-iqxsrc>, which is exactly zero for off-forwawrd case
phase_attached_to = "2pt"     #choose between "2pt" & "FF"


#the GEVP of each pz, used by all the cells below:  "basis" = its operators, "t0" = GEVP reference time, "t_ref" = time of the ground-state vector v0
#an operator is smear + gamma:  smear B = [4.5,2.25,0.8], U = [1.5,1.5,0];  gamma G5, G5G4, G3G5
gevp_choice = {0: {"basis": ["BG5", "BG5G4", "BG3G5", "UG5", "UG5G4", "UG3G5"], "t0": 3, "t_ref": 5},
               1: {"basis": ["BG3G5", "UG5G4",], "t0": 3, "t_ref": 5},
               2: {"basis": ["UG5G4", "UG3G5"], "t0": 3, "t_ref": 5},
               3: {"basis": ["UG5G4", "UG3G5"], "t0": 3, "t_ref": 5},
               4: {"basis": ["BG3G5","UG5G4"], "t0": 3, "t_ref": 5},
               5: {"basis": ["BG5G4", "UG3G5"], "t0": 3, "t_ref": 5},
               6: {"basis": ["BG3G5", "UG3G5"], "t0": 3, "t_ref": 5}}
gevp_tag = "basis1_t03_tref5"       #the name of this choice:  everything goes in the folder AI_assisted_matrix_element_analysis/{gevp_tag};  change it when you change gevp_choice
op_basis = [(0,0),(0,1),(0,2),(1,0),(1,1),(1,2)]    #the six operators of the files, (smear, gamma):  smear 0 = B, 1 = U;  gamma 0 = G5, 1 = G5G4, 2 = G3G5
smear_names = ["B", "U"]
gamma_names = ["G5", "G5G4", "G3G5"]
op_labels = [smear_names[s] + gamma_names[g] for s, g in op_basis]     #["BG5", "BG5G4", ...]

#directories and names
pt2_dir = "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta"
save_dir = os.path.join(pt2_dir, "AI_assisted_matrix_element_analysis", gevp_tag)
#C3input: q-phase averaged in;  C2pf, C2pi: no q-phase averaged in;  at q = 0 the C3input file is also the C2pf and the C2pi file
pt2_template = os.path.join(pt2_dir, "2pt_coulomb_rho4p5z2p25-rho1p5_frac0p8-0_smear_GEVP_{frame}_q{qx}_{qy}_{qz}_{pt2_type}_fb_tsep20_3072src_ncfg{ncfg}_{hadron}.h5")


assert frame in ["forward", "sym"], f"frame = {frame}: use forward or sym"
if frame == "forward":
    q_list = [(0,0,0)]
else:
    q_list = [(2,0,0), (-2,0,0), (0,2,0), (0,-2,0)]
q = tuple(q)
assert q in q_list, f"frame = {frame} with q = {q}: use one of {q_list}"
for pz in gevp_choice:
    assert gevp_choice[pz]["t_ref"] > gevp_choice[pz]["t0"], f"pz = {pz}: t_ref has to be larger than t0"
    for name in gevp_choice[pz]["basis"]:
        assert name in op_labels, f"pz = {pz}: {name} is not one of {op_labels}"

if hadron == "pion":
    hadron_tag = "pion"
elif project:
    hadron_tag = "proton_projected"
else:
    hadron_tag = "proton"
c_sign = {"pion": -1, "proton": +1}[hadron]      #to make Cij positive definite, the pion is multiplied by -1

with h5py.File(pt2_template.format(frame=frame, qx=q[0], qy=q[1], qz=q[2], pt2_type="C3input", ncfg=ncfg, hadron=hadron_tag), "r") as f:
    smear_list = f["smear_list"][()]             #(smear, 3): rho_T, rho_z, boost fraction


#======== functions used by several cells ========
def jk(data, jk_axis=0):
    #jackknife mean and error
    n = data.shape[jk_axis]
    data = np.moveaxis(data, jk_axis, 0)
    mean = np.mean(data, axis=0)
    err = np.sqrt((n - 1.0) / n * np.sum((data - mean) ** 2, axis=0))
    return mean, err


def jackknife(x):
    #per-cfg data (cfg, ...) -> jackknife samples (jk, ...)
    n_cfg = x.shape[0]
    return (x.sum(axis=0) - x) / (n_cfg - 1)


def read_pt2(q_i, pt2_type):
    #both halves of a packed 2pt file: (cfg, smear, smear, gamma, gamma, t_src, pz, tsep)
    pt2_file = pt2_template.format(frame=frame, qx=q_i[0], qy=q_i[1], qz=q_i[2], pt2_type=pt2_type, ncfg=ncfg, hadron=hadron_tag)
    print("reading", pt2_file)
    assert os.path.exists(pt2_file), f"no such file: {pt2_file}"
    with h5py.File(pt2_file, "r") as f:
        return f["pt2_forward"][()], f["pt2_backward"][()]


def solve_gevp(C_matrix, t0_pz):
    #one pz:  C_matrix (op, op, tsep): eigenvalues (state, tsep) and eigenvectors (state, tsep, op), largest eigenvalue first, NaN at tsep <= t0_pz
    n_op, _, n_tsep = C_matrix.shape
    eigen_val = np.full((n_op, n_tsep), np.nan)
    eigen_vec = np.full((n_op, n_tsep, n_op), np.nan, dtype=complex)
    L_inv = np.linalg.inv(np.linalg.cholesky(C_matrix[:, :, t0_pz]))
    for t in range(t0_pz + 1, n_tsep):
        M = L_inv @ C_matrix[:, :, t] @ L_inv.conj().T
        w, u = np.linalg.eigh(M)
        v = L_inv.conj().T @ u
        order = np.argsort(w)[::-1]
        eigen_val[:, t] = w[order]
        eigen_vec[:, t] = v[:, order].T
    return eigen_val, eigen_vec


def save_gevp_info(f, basis):
    #what every saved file states about the GEVP;  f is an open h5py file
    f["gevp_tag"] = gevp_tag
    f["gevp_basis"] = [" ".join(gevp_choice[pz]["basis"]) for pz in gevp_choice]    #(pz,): the operators of each pz in words
    f["op_basis"] = basis               #(op, 2): (smear, gamma) indices of smear_names, gamma_names
    f["smear_names"] = smear_names
    f["smear_list"] = smear_list
    f["gamma_names"] = gamma_names
    f["t0"] = [gevp_choice[pz]["t0"] for pz in gevp_choice]         #(pz,)
    f["t_ref"] = [gevp_choice[pz]["t_ref"] for pz in gevp_choice]   #(pz,)


def op_matrix(pt2, basis):
    #(cfg, smear_snk, smear_src, gamma_snk, gamma_src, ...) -> (cfg, op_snk, op_src, ...)
    n_op = len(basis)
    Cij = np.zeros(pt2.shape[:1] + (n_op, n_op) + pt2.shape[5:], dtype=complex)
    for a, (s_snk, g_snk) in enumerate(basis):
        for b, (s_src, g_src) in enumerate(basis):
            Cij[:, a, b] = pt2[:, s_snk, s_src, g_snk, g_src]
    return Cij


def apply_v0(Cij, v):
    #v^dagger C v:  Cij (cfg, op, op, ..., pz, tsep), v (pz, op) -> (cfg, ..., pz, tsep)
    return c_sign * np.einsum("pa,cab...pT,pb->c...pT", v.conj(), Cij, v)


def pt3_path(q_i):
    #the 3pt file of the basis of this cell at one q
    q_tag_i = f"q{q_i[0]}_{q_i[1]}_{q_i[2]}"
    if frame == "forward":
        folder = "forward"
    else:
        folder = f"sym_{q_tag_i}"
    return os.path.join(save_dir, "3pt_data", folder, f"3pt_{frame}_{q_tag_i}_{gevp_tag}_phase{phase_attached_to}_hyp{hyp}_ncfg{ncfg}_{hadron_tag}.h5")


gevp_file = os.path.join(save_dir, "2pt_gevp_data", frame, f"2pt_gevp_{frame}_{gevp_tag}_ncfg{ncfg}_{hadron_tag}.h5")

#---- the folder of this choice:  another gevp_choice with the same gevp_tag stops here, so two choices cannot get mixed in one folder
choice_text = ""
for pz in gevp_choice:
    choice_text = choice_text + f"pz {pz}: {' '.join(gevp_choice[pz]['basis'])}, t0 {gevp_choice[pz]['t0']}, t_ref {gevp_choice[pz]['t_ref']}\n"
os.makedirs(save_dir, exist_ok=True)
choice_file = os.path.join(save_dir, "gevp_choice.txt")
if not os.path.exists(choice_file):
    with open(choice_file, "w") as f:
        f.write(choice_text)
with open(choice_file, "r") as f:
    assert f.read() == choice_text, f"{choice_file} holds another gevp_choice:  change gevp_tag, or set gevp_choice back"
print("hadron", hadron_tag, "| frame", frame, "| q", q)
print("GEVP choice", gevp_tag, "| folder", save_dir)
print(choice_text)

hadron pion | frame forward | q (0, 0, 0)
GEVP choice basis1_t03_tref5 | folder /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/AI_assisted_matrix_element_analysis/basis1_t03_tref5
pz 0: BG5 BG5G4 BG3G5 UG5 UG5G4 UG3G5, t0 3, t_ref 5
pz 1: BG3G5 UG5G4, t0 3, t_ref 5
pz 2: UG5G4 UG3G5, t0 3, t_ref 5
pz 3: UG5G4 UG3G5, t0 3, t_ref 5
pz 4: BG3G5 UG5G4, t0 3, t_ref 5
pz 5: BG5G4 UG3G5, t0 3, t_ref 5
pz 6: BG3G5 UG3G5, t0 3, t_ref 5



In [10]:
#======== GEVP: reconstruct 2pt with the choice of cell 0 and save on disk ========
#every pz has its own operators, t0 and t_ref;  v0 is saved with all six operators, 0 for the ones this pz does not use

#---- read the 2pt of every q and average over t_src
if frame == "forward":
    pt2_type = "C3input"        #at q = 0 this file is also C2pf and C2pi
else:
    pt2_type = "C2pf"           #C2pi(q) is exactly C2pf(-q), so the C2pi files are not needed

half_list = ["forward", "backward"]
C2_raw = {"forward": [], "backward": []}    #C2_raw[half][iq]: (cfg, smear, smear, gamma, gamma, pz, tsep)
pf_list = []                                #pf_list[iq]: (pz, 3), the momentum of each 2pt
for q_i in q_list:
    raw_forward, raw_backward = read_pt2(q_i, pt2_type)
    assert raw_forward.ndim == 8, "the GEVP needs (cfg, smear, smear, gamma, gamma, t_src, pz, tsep): open Dirac files have to be projected first"
    C2_raw["forward"].append(raw_forward.mean(axis=5))  #here averaged over tsrc
    C2_raw["backward"].append(raw_backward.mean(axis=5))
    pt2_file_i = pt2_template.format(frame=frame, qx=q_i[0], qy=q_i[1], qz=q_i[2], pt2_type=pt2_type, ncfg=ncfg, hadron=hadron_tag)
    with h5py.File(pt2_file_i, "r") as f:
        pf_list.append(f["pf_list"][()])   #[()] reads in array and number
        pz_list = f["pz_list"][()]
        if q_i == q_list[0]:
            cfg_list = f["cfg_list"][()]
        assert np.array_equal(f["cfg_list"][()], cfg_list), f"q = {q_i}: other cfgs than the first q"
assert list(gevp_choice) == list(pz_list), f"gevp_choice has to have every pz of the files, in order: {list(pz_list)}"

n_cfg = C2_raw["forward"][0].shape[0]
n_pz = C2_raw["forward"][0].shape[5]
n_tsep = C2_raw["forward"][0].shape[6]
n_q = len(q_list)
n_all = len(op_basis)

#---- every half, q and pz: solve the GEVP on the operators of this pz, reconstruct the 2pt
C2_rec = {}
v0 = {}
eigen_val = {}
for half in half_list:
    C2_rec[half] = np.zeros((n_cfg, n_q, n_pz, n_tsep), dtype=complex)      #(cfg, q, pz, tsep)
    v0[half] = np.zeros((n_q, n_pz, n_all), dtype=complex)                  #(q, pz, op), all six operators
    eigen_val[half] = np.full((n_q, n_pz, n_all, n_tsep), np.nan)           #(q, pz, state, tsep), NaN for the states this pz does not have
    for iq, q_i in enumerate(q_list):
        iq_minus = q_list.index((-q_i[0], -q_i[1], -q_i[2]))
        C_plus = op_matrix(C2_raw[half][iq], op_basis)                      #(cfg, op, op, pz, tsep), +q (q/2,0,pz)
        C_minus = op_matrix(C2_raw[half][iq_minus], op_basis)               #(cfg, op, op, pz, tsep), -q (-q/2,0,pz)
        #gevp problem was solved for [C2(q) + C2(-q)] / 2, hermitianized: (op, op, pz, tsep)
        C_mean = c_sign * 0.5 * (C_plus + C_minus).mean(axis=0)
        C_mean = 0.5 * (C_mean + C_mean.conj().swapaxes(0, 1))
        for ipz, pz in enumerate(pz_list):
            use = [op_labels.index(name) for name in gevp_choice[pz]["basis"]]     #which of the six operators this pz uses
            C_use = C_mean[np.ix_(use, use)][:, :, ipz]                     #(n_use, n_use, tsep)
            val, vec = solve_gevp(C_use, gevp_choice[pz]["t0"])             #val (state, tsep), vec (state, tsep, op);  state 0 is the ground state
            eigen_val[half][iq, ipz, :len(use)] = val
            v0[half][iq, ipz, use] = vec[0, gevp_choice[pz]["t_ref"]]
        C2_rec[half][:, iq] = apply_v0(C_plus, v0[half][iq])

#---- save
os.makedirs(os.path.dirname(gevp_file), exist_ok=True)
with h5py.File(gevp_file, "w") as f:
    for half in half_list:
        f[f"C2_{half}"] = C2_rec[half]              #(cfg, q, pz, tsep), complex, c_sign already in;  q in the order of q_list
        f[f"v0_{half}"] = v0[half]                  #(q, pz, op), all six operators
        f[f"eigen_val_{half}"] = eigen_val[half]    #(q, pz, state, tsep), NaN at tsep <= t0
    f["q_list"] = q_list                            #(q, 3)
    f["pf_list"] = np.array(pf_list)                #(q, pz, 3): the momentum of C2
    f["pz_list"] = pz_list
    f["cfg_list"] = cfg_list
    save_gevp_info(f, op_basis)
print("saved", gevp_file)


reading /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/2pt_coulomb_rho4p5z2p25-rho1p5_frac0p8-0_smear_GEVP_forward_q0_0_0_C3input_fb_tsep20_3072src_ncfg250_pion.h5
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/AI_assisted_matrix_element_analysis/basis1_t03_tref5/2pt_gevp_data/forward/2pt_gevp_forward_basis1_t03_tref5_ncfg250_pion.h5


In [11]:
#======== 2pt fit: every fit parameter is here ========
q_average_or_not = True             #True = fit the average of the four q;  False = see single_q_data;  forward ignores this
single_q_data = "pair"              #only when q_average_or_not = False:  "pair" = average of q and -q,  "one" = q only;  q comes from cell 0
pt2_fit_half = "fb"                 #"forward", "backward" or "fb"
pt2_fit_window = {0: (3, 14), 1: (3, 14), 2: (3, 14), 3: (3, 14), 4: (3, 12), 5: (2, 10), 6: (2, 10)}    #pz: (first t, last t)
n_state = 3                         #number of states in the model
pt2_E0_center = "lattice"           #center of the E0 prior:  "lattice": cosh E = cosh m + sum(1 - cos p_i);  "continuum": E^2 = m^2 + sum(p_i^2);  p_i = 2 pi n_i / Ls
pt2_m_rest = {"pion": 0.1412, "proton": 0.4989}     #rest mass of each hadron in lattice units, you set it;  the center of the E0 prior comes from it
pt2_E0_width = 0.2                  #width of the E0 prior, the same at every pz
pt2_dE_prior = (0.25, 0.4)           #(center, width) of every gap at every pz;  the fit varies log(dE), so dE stays positive
pt2_c_prior = (0.0, 1e4)            #(center, width) of every c_n
pt2_backward_wave = True            #True = keep the term e^{-E (Lt - t)} in the model
pt2_correlated = True               #True = full covariance;  False = only its diagonal
pt2_svdcut = 1e-7
pt2_maxit = 20000
pt2_tol = 1e-8                      #stopping tolerance of the fit
pt2_fitter = "scipy_least_squares"  #the minimizer
pt2_fit_label = ""                  #extra text in the file name;  set it when you change a prior, svdcut, ..., or the old file is replaced

#---- check the settings
print("GEVP choice", gevp_tag)
assert pt2_fit_half in ["forward", "backward", "fb"], f"pt2_fit_half = {pt2_fit_half}: use forward, backward or fb"
assert single_q_data in ["pair", "one"], f"single_q_data = {single_q_data}: use pair or one"
assert pt2_E0_center in ["lattice", "continuum"], f"pt2_E0_center = {pt2_E0_center}: use lattice or continuum"
assert n_state >= 1, f"n_state = {n_state}: use at least 1"
assert hadron in pt2_m_rest, f"no mass for {hadron} in pt2_m_rest"
assert os.path.exists(gevp_file), f"no GEVP file for {gevp_tag}: run cell 1 first"

#---- read the reconstructed 2pt of this basis
C2_saved = {}
with h5py.File(gevp_file, "r") as f:
    C2_saved["forward"] = f["C2_forward"][()].real      #(cfg, q, pz, tsep)
    C2_saved["backward"] = f["C2_backward"][()].real    #(cfg, q, pz, tsep)
    pf_saved = f["pf_list"][()]                         #(q, pz, 3)
    pz_list = f["pz_list"][()]
    cfg_list = f["cfg_list"][()]
for pz in pz_list:
    assert pz in pt2_fit_window, f"pz = {pz} is missing from pt2_fit_window"

#If it's forward, we have only one type of 2pt, so just fit that one
#If it's off-forward, we could fit for example the average of (1,0,pz) and (-1,0,pz) or the average of all four since they should be the same
iq = q_list.index(q)
iq_minus = q_list.index((-q[0], -q[1], -q[2]))
if frame == "forward":
    iq_use = [0]
    pt2_q_tag = "q0_0_0"
    pt2_subfolder = "forward"
elif q_average_or_not:
    iq_use = [0, 1, 2, 3]
    pt2_q_tag = "averaged_over_q"
    pt2_subfolder = "sym_averaged_over_q"
elif single_q_data == "pair":
    iq_use = [iq, iq_minus]
    pt2_q_tag = f"q{q[0]}_{q[1]}_{q[2]}"
    pt2_subfolder = f"sym_{pt2_q_tag}"
else:
    iq_use = [iq]
    pt2_q_tag = f"q{q[0]}_{q[1]}_{q[2]}_one"
    pt2_subfolder = f"sym_{pt2_q_tag}"

#---- the data of the fit: average over the halves and the q of each cfg, then jackknife
if pt2_fit_half == "fb":
    C2_half = 0.5 * (C2_saved["forward"] + C2_saved["backward"])        #(cfg, q, pz, tsep)
else:
    C2_half = C2_saved[pt2_fit_half]                                    #(cfg, q, pz, tsep)

#Use fancy indexing to average over q's
pt2_fit_jk = jackknife(C2_half[:, iq_use].mean(axis=1))                 #(jk, pz, tsep)

#calculate central of prior based on lattice/continuous dispersion relation
m_rest = pt2_m_rest[hadron]
p_i = 2 * np.pi * pf_saved[iq_use[0]] / Ls                              #(pz, 3)
if pt2_E0_center == "lattice":
    E0_center = np.arccosh(np.cosh(m_rest) + (1 - np.cos(p_i)).sum(axis=1))     #(pz,)
else:
    E0_center = np.sqrt(m_rest ** 2 + (p_i ** 2).sum(axis=1))                   #(pz,)

#model and prior
def pt2_model(t, p):
    #sum over the states of c_n (e^{-E_n t} + e^{-E_n (Lt - t)}),  E_n = E0 + dE_1 + ... + dE_n
    E = p["E0"]
    corr = 0
    for n in range(n_state):
        if n > 0:
            E = E + p["dE"][n - 1]
        if pt2_backward_wave:
            wave = gv.exp(-E * t) + gv.exp(-E * (Lt - t))
        else:
            wave = gv.exp(-E * t)
        corr = corr + p["c"][n] * wave
    return corr


def pt2_prior(ipz):
    #returns the prior dictionary of ipz
    #calculate E0 center from dispersion relation
    #width directly set at the beginning
    prior = gv.BufferDict()
    prior["E0"] = gv.gvar(E0_center[ipz], pt2_E0_width)
    c_center = []
    c_width = []
    for n in range(n_state):
        c_center.append(pt2_c_prior[0])
        c_width.append(pt2_c_prior[1])
    prior["c"] = gv.gvar(c_center, c_width)     #(n_state,)
    if n_state > 1:
        prior["log(dE)"] = gv.log(gv.gvar(np.full(n_state - 1, float(pt2_dE_prior[0])), np.full(n_state - 1, float(pt2_dE_prior[1]))))
    return prior


#fit central value first and use the central fit's result as the starting point of jackknife fit.
n_jk = pt2_fit_jk.shape[0]
n_pz = len(pz_list)
pt2_E_jk = np.zeros((n_jk, n_pz, n_state))          #(jk, pz, state): E_n
pt2_c_jk = np.zeros((n_jk, n_pz, n_state))          #(jk, pz, state): c_n
pt2_E_central = np.zeros((n_pz, n_state))           #(pz, state): the fit of the jackknife mean
pt2_E_lsqfit_err = np.zeros((n_pz, n_state))        #(pz, state): its lsqfit error, which sees the priors
pt2_c_central = np.zeros((n_pz, n_state))           #(pz, state)
pt2_c_lsqfit_err = np.zeros((n_pz, n_state))        #(pz, state)
pt2_chi2dof = np.zeros(n_pz)                        #(pz,): the fit of the jackknife mean
pt2_Q = np.zeros(n_pz)                              #(pz,)
pt2_fit_result = {}
pt2_n_not_converged = 0                             #how many fits stopped at pt2_maxit before converging;  lsqfit gives no warning for this
gv.switch_gvar()        #stops memory from growing forever
for ipz, pz in enumerate(pz_list):
    first, last = pt2_fit_window[pz]
    t = np.arange(first, last + 1)                  #(n_t,)
    assert len(t) > n_state + 1, f"pz = {pz}: {len(t)} points for {n_state + 1} free parameters, make the window longer"
    y_jk = pt2_fit_jk[:, ipz, t]                    #(jk, n_t)

    #bias gives a factor of 1/Njk so multiply by (nik-1) to match formula
    cov = (n_jk - 1) * np.cov(y_jk, rowvar=False, bias=True)    #(n_t, n_t): jackknife covariance
    
    if not pt2_correlated:
        #if uncorrelated, trim the covariance matrix to diagonal 
        cov = np.diag(np.diag(cov))

    fit = lsqfit.nonlinear_fit(data=(t, gv.gvar(y_jk.mean(axis=0), cov)), fcn=pt2_model, prior=pt2_prior(ipz), svdcut=pt2_svdcut, maxit=pt2_maxit, tol=pt2_tol, fitter=pt2_fitter)
    pt2_fit_result[pz] = fit
    if fit.stopping_criterion == 0:
        pt2_n_not_converged = pt2_n_not_converged + 1
    E_n = fit.p["E0"]
    for n in range(n_state):
        if n > 0:
            E_n = E_n + fit.p["dE"][n - 1]
        pt2_E_central[ipz, n] = E_n.mean
        pt2_E_lsqfit_err[ipz, n] = E_n.sdev
        pt2_c_central[ipz, n] = fit.p["c"][n].mean
        pt2_c_lsqfit_err[ipz, n] = fit.p["c"][n].sdev
    dof = fit.dof      #
    #print(dof)
    print(fit)
    print(fit.chi2/fit.dof)
    pt2_chi2dof[ipz] = fit.chi2 / dof
    pt2_Q[ipz] = fit.Q

    for i in range(n_jk):
        #print(i)
        fit_i = lsqfit.nonlinear_fit(data=(t, gv.gvar(y_jk[i], cov)), fcn=pt2_model, prior=pt2_prior(ipz), p0=fit.pmean, svdcut=pt2_svdcut, maxit=pt2_maxit, tol=pt2_tol, fitter=pt2_fitter)
        if fit_i.stopping_criterion == 0:
            pt2_n_not_converged = pt2_n_not_converged + 1
        E_i = fit_i.pmean["E0"]
        for n in range(n_state):
            if n > 0:
                #here, even though we used log(dE) in the prior, gvar knows dE, log() is overloaded in gvar, it knows this function
                E_i = E_i + fit_i.pmean["dE"][n - 1]
            pt2_E_jk[i, ipz, n] = E_i
            pt2_c_jk[i, ipz, n] = fit_i.pmean["c"][n]
gv.restore_gvar()

#---- print the result: jk = jackknife mean and error, lsqfit = the fit of the mean, its error sees the priors
print(f"{hadron_tag}, {frame}, {pt2_q_tag}, {pt2_fit_half}, {n_state} states, rest mass {m_rest:.4f}")
print(f"{'pz':>3s} {'t':>5s} {'E0 jk':>11s} {'E0 lsqfit':>11s} {'center':>8s} {'dE1 jk':>10s} {'dE1 lsqfit':>11s} {'c1/c0 jk':>10s} {'chi2/dof':>9s} {'Q':>6s}")
for ipz, pz in enumerate(pz_list):
    first, last = pt2_fit_window[pz]
    mean, err = jk(pt2_E_jk[:, ipz, 0])
    E0_text = str(gv.gvar(mean, err))
    E0_lsqfit_text = str(gv.gvar(pt2_E_central[ipz, 0], pt2_E_lsqfit_err[ipz, 0]))
    if n_state > 1:
        mean, err = jk(pt2_E_jk[:, ipz, 1] - pt2_E_jk[:, ipz, 0])
        dE1_text = str(gv.gvar(mean, err))
        dE1_lsqfit_text = str(pt2_fit_result[pz].p["dE"][0])
        mean, err = jk(pt2_c_jk[:, ipz, 1] / pt2_c_jk[:, ipz, 0])
        c_ratio_text = str(gv.gvar(mean, err))
    else:
        dE1_text = "-"
        dE1_lsqfit_text = "-"
        c_ratio_text = "-"
    print(f"{pz:3d} {first:2d}-{last:2d} {E0_text:>11s} {E0_lsqfit_text:>11s} {E0_center[ipz]:8.4f} {dE1_text:>10s} {dE1_lsqfit_text:>11s} {c_ratio_text:>10s} {pt2_chi2dof[ipz]:9.2f} {pt2_Q[ipz]:6.3f}")
print(f"fits that stopped at pt2_maxit before converging: {pt2_n_not_converged} of {n_pz * (n_jk + 1)}")

#---- save: the file states the fitted data and every setting
pt2_range_tag = "_".join(f"{pt2_fit_window[pz][0]}-{pt2_fit_window[pz][1]}n{n_state}" for pz in pz_list)     #first t - last t, number of states, one per pz
if pt2_fit_label == "":
    pt2_label_text = ""
else:
    pt2_label_text = "_" + pt2_fit_label
pt2_fit_file = os.path.join(save_dir, "2pt_fit_data", pt2_subfolder, f"2pt_fit_{frame}_{pt2_q_tag}_{gevp_tag}_t{pt2_range_tag}_{pt2_fit_half}_ncfg{ncfg}_svd{pt2_svdcut:g}{pt2_label_text}_{hadron_tag}.h5")
if os.path.exists(pt2_fit_file):
    print("replacing", pt2_fit_file)
os.makedirs(os.path.dirname(pt2_fit_file), exist_ok=True)
with h5py.File(pt2_fit_file, "w") as f:
    #results
    f["fit_data_jk"] = pt2_fit_jk                           #(jk, pz, tsep): the data, for the plots
    f["E_jk"] = pt2_E_jk                                    #(jk, pz, state)
    f["c_jk"] = pt2_c_jk                                    #(jk, pz, state)
    f["E_central"] = pt2_E_central                          #(pz, state)
    f["E_lsqfit_err"] = pt2_E_lsqfit_err                    #(pz, state)
    f["c_central"] = pt2_c_central                          #(pz, state)
    f["c_lsqfit_err"] = pt2_c_lsqfit_err                    #(pz, state)
    f["chi2dof"] = pt2_chi2dof                              #(pz,)
    f["Q"] = pt2_Q                                          #(pz,)
    f["n_not_converged"] = pt2_n_not_converged
    #settings
    f["q_average_or_not"] = q_average_or_not
    f["single_q_data"] = single_q_data
    f["q_tag"] = pt2_q_tag
    f["subfolder"] = pt2_subfolder
    f["q_list"] = [q_list[i] for i in iq_use]               #(n_q, 3): the q that went in
    f["fit_half"] = pt2_fit_half
    f["fit_window"] = [pt2_fit_window[pz] for pz in pz_list]    #(pz, 2): first t, last t
    f["fit_range_tag"] = pt2_range_tag
    f["n_state"] = n_state
    f["E0_center_kind"] = pt2_E0_center
    f["E0_prior_center"] = E0_center                        #(pz,)
    f["E0_prior_width"] = pt2_E0_width
    f["m_rest"] = m_rest
    f["dE_prior"] = pt2_dE_prior                            #(center, width)
    f["c_prior"] = pt2_c_prior                              #(center, width)
    f["backward_wave"] = pt2_backward_wave
    f["correlated"] = pt2_correlated
    f["svdcut"] = pt2_svdcut
    f["maxit"] = pt2_maxit
    f["tol"] = pt2_tol
    f["fitter"] = pt2_fitter
    f["fit_label"] = pt2_fit_label
    #what the data are
    f["frame"] = frame
    f["hadron_tag"] = hadron_tag
    f["ncfg"] = ncfg
    f["cfg_list"] = cfg_list
    f["pz_list"] = pz_list
    f["pf_list"] = pf_saved[iq_use[0]]                      #(pz, 3)
    f["Ls"] = Ls
    f["Lt"] = Lt
    save_gevp_info(f, op_basis)
print("saved", pt2_fit_file)

GEVP choice basis1_t03_tref5
Least Squares Fit:
  chi2/dof [dof] = 0.22 [12]    Q = 1    logGBF = 49.422

Parameters:
             E0   0.14110 (19)      [   0.14 (20) ]  
            c 0    1.5130 (29)      [ 0 ± 1.0e+04 ]  
              1     0.043 (90)      [ 0 ± 1.0e+04 ]  
              2      1.3 (2.0)      [ 0 ± 1.0e+04 ]  
      log(dE) 0     -0.43 (62)      [  -1.4 (1.6) ]  
              1      0.06 (38)      [  -1.4 (1.6) ]  
---------------------------------------------------
           dE 0      0.65 (40)      [   0.25 (40) ]  
              1      1.06 (41)      [   0.25 (40) ]  *

Settings:
  svdcut/n = 1e-07/0    tol = (1e-08,1e-10,1e-10*)    (itns/time = 84/0.0s)
  fitter = scipy_least_squares    method = trf

0.224960470707501
Least Squares Fit:
  chi2/dof [dof] = 0.86 [12]    Q = 0.59    logGBF = 45.397

Parameters:
             E0   0.24168 (45)      [   0.24 (20) ]  
            c 0    1.9581 (71)      [ 0 ± 1.0e+04 ]  
              1      0.8 (1.7)      [ 0 ± 1.

In [12]:
#======== plot the saved 2pt fit: effective mass, one figure per pz, saved in meff_plots/{forward, sym_q..., sym_averaged_over_q} ========
#points: log[C(t) / C(t + 1)] at t + 1/2, filled inside the fit window, open outside;  blue band: the same of the fit curve of every jackknife sample
#gray band: E0;  dashed line: the center of the E0 prior
pt2_plot_file = pt2_fit_file        #the file cell 2 just saved;  in a new kernel paste here the path that cell 2 printed, as a string
meff_plot_t = np.arange(2, 19)      #the t of log[C(t) / C(t + 1)]
meff_plot_err_max = 0.1             #leave out points with a larger error, so noisy t do not set the y range
meff_color = "#2a78d6"
meff_shift=0

#---- read the saved fit
with h5py.File(pt2_plot_file, "r") as f:
    data_jk = f["fit_data_jk"][()]                      #(jk, pz, tsep)
    E_jk = f["E_jk"][()]                                #(jk, pz, state)
    c_jk = f["c_jk"][()]                                #(jk, pz, state)
    E_central = f["E_central"][()]                      #(pz, state)
    E_lsqfit_err = f["E_lsqfit_err"][()]                #(pz, state)
    chi2dof = f["chi2dof"][()]                          #(pz,)
    Q = f["Q"][()]                                      #(pz,)
    window = f["fit_window"][()]                        #(pz, 2)
    E0_prior_center = f["E0_prior_center"][()]          #(pz,)
    plot_pz_list = f["pz_list"][()]
    plot_n_state = f["n_state"][()]
    plot_Lt = f["Lt"][()]
    if f["backward_wave"][()]:
        backward_term = 1
    else:
        backward_term = 0
    plot_half = f["fit_half"].asstr()[()]
    plot_q_tag = f["q_tag"].asstr()[()]
    plot_subfolder = f["subfolder"].asstr()[()]
    plot_frame = f["frame"].asstr()[()]
    plot_hadron = f["hadron_tag"].asstr()[()]
    plot_ncfg = f["ncfg"][()]
    plot_gevp_tag = f["gevp_tag"].asstr()[()]
    plot_label = f["fit_label"].asstr()[()]
    plot_svdcut = f["svdcut"][()]
    plot_t0 = f["t0"][()]
    plot_t_ref = f["t_ref"][()]
    plot_basis = list(f["gevp_basis"].asstr()[()])

if plot_label == "":
    plot_label_text = ""
else:
    plot_label_text = "_" + plot_label
meff_plot_dir = os.path.join(save_dir, "meff_plots", plot_subfolder)
os.makedirs(meff_plot_dir, exist_ok=True)

#---- one figure per pz
for ipz, pz in enumerate(plot_pz_list):
    first, last = window[ipz]
    C_jk = data_jk[:, ipz]                              #(jk, tsep)
    with np.errstate(invalid="ignore"):                 #a negative C at a noisy t gives NaN, those points are dropped below
        m, e = jk(np.log(C_jk[:, meff_plot_t] / C_jk[:, meff_plot_t + 1]))
    m = np.where(e < meff_plot_err_max, m, np.nan)
    in_fit = (meff_plot_t >= first) & (meff_plot_t < last)      #both C(t) and C(t + 1) are in the fit

    #the band: the model at t and t + 1 for every jackknife sample
    t_band = np.linspace(first, last - 1, 60)           #(n_t,)
    E_n = E_jk[:, ipz, :, None]                         #(jk, state, 1)
    c_n = c_jk[:, ipz, :, None]                         #(jk, state, 1)
    C_t = (c_n * (np.exp(-E_n * t_band) + backward_term * np.exp(-E_n * (plot_Lt - t_band)))).sum(axis=1)                   #(jk, n_t)
    C_t1 = (c_n * (np.exp(-E_n * (t_band + 1)) + backward_term * np.exp(-E_n * (plot_Lt - t_band - 1)))).sum(axis=1)        #(jk, n_t)
    band_mean, band_err = jk(np.log(C_t / C_t1))

    mean, err = jk(E_jk[:, ipz, 0])
    E0 = gv.gvar(mean, err)
    E0_lsqfit = gv.gvar(E_central[ipz, 0], E_lsqfit_err[ipz, 0])
    if plot_n_state > 1:
        mean, err = jk(E_jk[:, ipz, 1] - E_jk[:, ipz, 0])
        dE1_text = f"dE1 = {gv.gvar(mean, err)}, "
    else:
        dE1_text = ""

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.axhspan(E0.mean - E0.sdev, E0.mean + E0.sdev, color="#e7e7e5", lw=0, zorder=0, label="E0")
    ax.axhline(E0_prior_center[ipz], color="#52514e", lw=1, ls="--", zorder=1, label="E0 prior center")
    
    ax.fill_between(t_band + meff_shift, band_mean - band_err, band_mean + band_err, color=meff_color, alpha=0.25, lw=0, zorder=1, label="fit")
    ax.errorbar(meff_plot_t[in_fit] + meff_shift, m[in_fit], e[in_fit], color=meff_color, marker="o", ms=5, ls="none", elinewidth=1, capsize=2, zorder=2, label="in the fit")
    ax.errorbar(meff_plot_t[~in_fit] + meff_shift, m[~in_fit], e[~in_fit], color=meff_color, marker="o", mfc="white", ms=5, ls="none", elinewidth=1, capsize=2, zorder=2, label="not in the fit")
    
    #y range from the points in the fit, so the small t do not squash the plateau
    y_low = np.nanmin(np.append(m[in_fit] - e[in_fit], [E0.mean - E0.sdev, E0_prior_center[ipz]]))
    y_high = np.nanmax(np.append(m[in_fit] + e[in_fit], [E0.mean + E0.sdev, E0_prior_center[ipz]]))
    ax.set_ylim(y_low - 0.1 * (y_high - y_low), y_high + 0.1 * (y_high - y_low))
    ax.grid(color="#e1e0d9", lw=0.6)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel("t")
    ax.set_ylabel(r"$\log[C(t - 1/2) / C(t + 1/2)]$")
    ax.set_title(f"{plot_hadron}, {plot_frame}, {plot_q_tag}, {plot_half}, pz = {pz}, {plot_ncfg} cfg\n"
                 f"{plot_basis[ipz]}, t0 {plot_t0[ipz]}, t_ref {plot_t_ref[ipz]}\n"
                 f"t {first}-{last}, {plot_n_state} states: E0 = {E0} (lsqfit {E0_lsqfit}), {dE1_text}chi2/dof {chi2dof[ipz]:.2f}, Q {Q[ipz]:.3f}", fontsize=9)
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)
    fig.tight_layout()
    fig.savefig(os.path.join(meff_plot_dir, f"meff_{plot_frame}_{plot_q_tag}_{plot_gevp_tag}_t{first}-{last}n{plot_n_state}_{plot_half}_ncfg{plot_ncfg}_svd{plot_svdcut:g}{plot_label_text}_{plot_hadron}_pz{pz}.png"), dpi=150)
    plt.close(fig)
print("saved", len(plot_pz_list), "plots in", meff_plot_dir)

saved 7 plots in /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/AI_assisted_matrix_element_analysis/basis1_t03_tref5/meff_plots/forward


In [13]:
#======== 3pt, method 1: one file per q ========
pt3_q_list = q_list         #the q to make;  q_list = all of them, or for example [(2,0,0)]
tau_list = np.arange(20)
half_sign = {"forward": +1, "backward": -1}     #operator at t_src + tau (forward), t_src - tau (backward)

print("GEVP choice", gevp_tag)
assert phase_attached_to in ["FF", "2pt"], f"phase_attached_to = {phase_attached_to}: use FF or 2pt"
assert os.path.exists(gevp_file), f"no GEVP file for {gevp_tag}: run cell 1 first"

#---- the saved 2pt and v0 of this basis
C2_saved = {}
v0_saved = {}
with h5py.File(gevp_file, "r") as f:
    for half in half_sign:
        C2_saved[half] = f[f"C2_{half}"][()]    #(cfg, q, pz, tsep)
        v0_saved[half] = f[f"v0_{half}"][()]    #(q, pz, op)

for q_i in pt3_q_list:
    print("3pt at q =", q_i)
    iq = q_list.index(q_i)
    iq_minus = q_list.index((-q_i[0], -q_i[1], -q_i[2]))

    #---- source positions, and C3input = v^dagger C v with the saved v0
    C3input_file = pt2_template.format(frame=frame, qx=q_i[0], qy=q_i[1], qz=q_i[2], pt2_type="C3input", ncfg=ncfg, hadron=hadron_tag)
    with h5py.File(C3input_file, "r") as f:
        cfg_list = f["cfg_list"][()]
        pz_list = f["pz_list"][()]
        t_src_list = f["t_src_list"][()]        #(cfg, t_src): the slice of each t_src index, not sorted, shifts per cfg
        x_src = f["x_src_list"][()]             #(cfg, 4): source positions, they shift per cfg
        y_src = f["y_src_list"][()]             #(cfg, 4)
        z_src = f["z_src_list"][()]             #(cfg, 8)
    raw_forward, raw_backward = read_pt2(q_i, "C3input")
    C3input = {}                                #C3input[half]: (cfg, t_src, pz, tsep), c_sign already in
    C3input["forward"] = apply_v0(op_matrix(raw_forward, op_basis), v0_saved["forward"][iq])
    C3input["backward"] = apply_v0(op_matrix(raw_backward, op_basis), v0_saved["backward"][iq])

    #mean of exp(-i 2pi/Ls q.x_src) over the 128 source positions, one number per cfg: 1 at q = 0, ~1e-16 at the four sym q
    phase_mean = (np.exp(-2j * np.pi * q_i[0] * x_src / Ls).mean(axis=1)
                  * np.exp(-2j * np.pi * q_i[1] * y_src / Ls).mean(axis=1)
                  * np.exp(-2j * np.pi * q_i[2] * z_src / Ls).mean(axis=1))

    #---- the gluon operator: FF (cfg, munu, rhosig, hyp, w, t), munu order XY, YZ, XZ, XT, YT, ZT
    FF_file = os.path.join(pt2_dir, f"FF_hyp0-8_w0-9_q{q_i[0]}_{q_i[1]}_{q_i[2]}_ncfg800.h5")
    with h5py.File(FF_file, "r") as f:
        assert tuple(f["q"][()]) == q_i
        FF_cfg = f["cfg_list"][()]
        idx = np.searchsorted(FF_cfg, cfg_list)     #the 2pt cfgs are not the first 250 FF cfgs
        assert np.array_equal(FF_cfg[idx], cfg_list)
        ihyp = list(f["hyp_list"][()]).index(hyp)
        w_list = f["w_list"][()]
        TXTX = f["FF"][idx, 3, 3, ihyp]
        TYTY = f["FF"][idx, 4, 4, ihyp]
        XYXY = f["FF"][idx, 0, 0, ihyp]
    O = TXTX + TYTY - 2 * XYXY                      #(cfg, w, t)

    #---- connected minus vacuum, for every jackknife sample
    n_cfg, n_tsrc = t_src_list.shape
    i_cfg = np.arange(n_cfg)[:, None, None]
    pt3_jk = {}
    C2pf_jk = {}
    C2pi_jk = {}
    for half in half_sign:
        C2pf_cfg = C2_saved[half][:, iq]            #(cfg, pz, tsep): plain 2pt at p_f = q/2
        C2pi_cfg = C2_saved[half][:, iq_minus]      #(cfg, pz, tsep): plain 2pt at p_i = -q/2, C2pi(q) = C2pf(-q)
        t_op = (t_src_list[:, :, None] + half_sign[half] * tau_list) % Lt       #(cfg, t_src, tau)
        O_tau = O[i_cfg, :, t_op]                   #(cfg, t_src, tau, w): numpy puts w last here
        connected = np.einsum("cspT,cstw->cwpTt", C3input[half], O_tau, optimize=True) / n_tsrc     #(cfg, w, pz, tsep, tau)
        if phase_attached_to == "FF":
            pt2_avg = C2pf_cfg                      #(cfg, pz, tsep), plain, no source phase
            op_avg = O_tau.mean(axis=1) * phase_mean[:, None, None]     #(cfg, tau, w), the source phase is here
        else:
            pt2_avg = C3input[half].mean(axis=1)    #(cfg, pz, tsep), the source phase is here
            op_avg = O_tau.mean(axis=1)             #(cfg, tau, w), no source phase
        pt3_jk[half] = jackknife(connected) - np.einsum("cpT,ctw->cwpTt", jackknife(pt2_avg), jackknife(op_avg), optimize=True)    #(jk, w, pz, tsep, tau)
        C2pf_jk[half] = jackknife(C2pf_cfg).real    #(jk, pz, tsep)
        C2pi_jk[half] = jackknife(C2pi_cfg).real    #(jk, pz, tsep)

    #check: C3/C2 at tsep 8, tau 4, w 0
    for half in half_sign:
        m, e = jk(pt3_jk[half][:, 0, :, 8, 4].real / C2pf_jk[half][:, :, 8])
        print(half, " ".join(f"{a:+.3f}({b * 1e3:.0f})" for a, b in zip(m, e)))

    #---- save the 3pt, with the jackknife 2pt that goes with it
    os.makedirs(os.path.dirname(pt3_path(q_i)), exist_ok=True)
    with h5py.File(pt3_path(q_i), "w") as f:
        for half in half_sign:
            f[f"pt3_{half}"] = pt3_jk[half]         #(jk, w, pz, tsep, tau), complex
            f[f"C2pf_{half}"] = C2pf_jk[half]       #(jk, pz, tsep)
            f[f"C2pi_{half}"] = C2pi_jk[half]       #(jk, pz, tsep)
        f["q"] = q_i
        f["cfg_list"] = cfg_list
        f["pz_list"] = pz_list
        f["w_list"] = w_list
        f["tsep_list"] = np.arange(pt3_jk["forward"].shape[3])
        f["tau_list"] = tau_list
        f["hyp"] = hyp
        save_gevp_info(f, op_basis)
        f["phase_attached_to"] = phase_attached_to
    print("saved", pt3_path(q_i))

GEVP choice basis1_t03_tref5
3pt at q = (0, 0, 0)
reading /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/2pt_coulomb_rho4p5z2p25-rho1p5_frac0p8-0_smear_GEVP_forward_q0_0_0_C3input_fb_tsep20_3072src_ncfg250_pion.h5
forward -0.145(22) -0.300(32) -0.419(39) -0.549(62) -0.295(17) -0.323(39) -0.240(32)
backward -0.159(22) -0.296(33) -0.395(37) -0.436(59) -0.288(18) -0.319(40) -0.245(30)
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/AI_assisted_matrix_element_analysis/basis1_t03_tref5/3pt_data/forward/3pt_forward_q0_0_0_basis1_t03_tref5_phase2pt_hyp8_ncfg250_pion.h5


In [14]:
#======== ratio, and its plots: one figure per w and pz ========
#R(tsep, tau) = C3(tsep, tau) / C2pf(tsep) * sqrt[ C2pi(tsep - tau) C2pf(tau) C2pf(tsep) / (C2pf(tsep - tau) C2pi(tau) C2pi(tsep)) ]
#the ground state gives M / (2 sqrt(E_f E_i)) at every tsep, tau;  at q = 0 the square root is 1
ratio_q_average = True                      #sym only: True = average the 3pt and 2pt over the four q, then make the ratio;  False = the q of cell 0
plot_half = "fb"                            #"forward", "backward" or "fb"
plot_tsep_list = [4,5,6,7,8,9,10,11,12,13,14,15]        #up to 8 values
plot_tau_skip = 1                           #leave out tau < plot_tau_skip and tau > tsep - plot_tau_skip
plot_err_max = 0.3                          #leave out points with a larger error, so noisy tsep do not set the y range

print("GEVP choice", gevp_tag)
half_list = ["forward", "backward"]

#---- which q go in, and the label of this ratio
if frame == "forward":
    ratio_q_list = [(0,0,0)]
    ratio_q_tag = "q0_0_0"
    ratio_subfolder = "forward"
elif ratio_q_average:
    ratio_q_list = q_list
    ratio_q_tag = "averaged_over_q"
    ratio_subfolder = "sym_averaged_over_q"
else:
    ratio_q_list = [q]
    ratio_q_tag = f"q{q[0]}_{q[1]}_{q[2]}"
    ratio_subfolder = f"sym_{ratio_q_tag}"

#---- read the 3pt files and average over q
pt3_use = {"forward": 0, "backward": 0}     #pt3_use[half]: (jk, w, pz, tsep, tau)
c2f_jk = {"forward": 0, "backward": 0}      #c2f_jk[half]: (jk, pz, tsep)
c2i_jk = {"forward": 0, "backward": 0}      #c2i_jk[half]: (jk, pz, tsep)
for q_i in ratio_q_list:
    assert os.path.exists(pt3_path(q_i)), f"no 3pt file at q = {q_i} for {gevp_tag}: run cell 4 first"
    with h5py.File(pt3_path(q_i), "r") as f:
        if q_i == ratio_q_list[0]:
            cfg_list = f["cfg_list"][()]
            pz_list = f["pz_list"][()]
            w_list = f["w_list"][()]
            tsep_list = f["tsep_list"][()]
            tau_list = f["tau_list"][()]
        assert np.array_equal(f["cfg_list"][()], cfg_list), f"q = {q_i}: other cfgs, the jackknife samples would not match"
        for half in half_list:
            pt3_use[half] = pt3_use[half] + f[f"pt3_{half}"][()] / len(ratio_q_list)
            c2f_jk[half] = c2f_jk[half] + f[f"C2pf_{half}"][()] / len(ratio_q_list)
            c2i_jk[half] = c2i_jk[half] + f[f"C2pi_{half}"][()] / len(ratio_q_list)

#---- the ratio
tsep_grid, tau_grid = np.meshgrid(tsep_list, tau_list, indexing="ij")           #(tsep, tau)
in_window = tau_grid <= tsep_grid
rest_grid = np.where(in_window, tsep_grid - tau_grid, 0)                        #tsep - tau, the 0 is only used where R is set to NaN


def build_ratio(pt3, c2f, c2i):
    #pt3 (jk, w, pz, tsep, tau), c2f and c2i real (jk, pz, tsep) -> R (jk, w, pz, tsep, tau), complex, NaN for tau > tsep
    root = np.sqrt(c2i[:, :, rest_grid] * c2f[:, :, tau_grid] * c2f[:, :, tsep_grid]
                   / (c2f[:, :, rest_grid] * c2i[:, :, tau_grid] * c2i[:, :, tsep_grid]))    #(jk, pz, tsep, tau)
    return np.where(in_window, pt3 * (root / c2f[:, :, tsep_grid])[:, None], np.nan)


ratio_jk = {}
for half in half_list:
    ratio_jk[half] = build_ratio(pt3_use[half], c2f_jk[half], c2i_jk[half])
#forward and backward together: average the jackknife 3pt and 2pt first, then the ratio
ratio_jk["fb"] = build_ratio(0.5 * (pt3_use["forward"] + pt3_use["backward"]),
                             0.5 * (c2f_jk["forward"] + c2f_jk["backward"]),
                             0.5 * (c2i_jk["forward"] + c2i_jk["backward"]))

#check: Re R at tsep 8, tau 4, w 0
for half in ratio_jk:
    m, e = jk(ratio_jk[half][:, 0, :, 8, 4].real)
    print(f"{half:8s}", " ".join(f"{a:+.3f}({b * 1e3:.0f})" for a, b in zip(m, e)))

#---- save the ratio
ratio_file = os.path.join(save_dir, "ratio_data", ratio_subfolder, f"ratio_{frame}_{ratio_q_tag}_{gevp_tag}_phase{phase_attached_to}_hyp{hyp}_ncfg{ncfg}_{hadron_tag}.h5")
os.makedirs(os.path.dirname(ratio_file), exist_ok=True)
with h5py.File(ratio_file, "w") as f:
    for half in ratio_jk:
        f[f"ratio_{half}"] = ratio_jk[half]         #(jk, w, pz, tsep, tau), complex, NaN for tau > tsep;  half = forward, backward, fb
    f["q_list"] = ratio_q_list                      #(n_q, 3): the q that went in
    f["q_tag"] = ratio_q_tag
    f["subfolder"] = ratio_subfolder
    f["frame"] = frame
    f["hadron_tag"] = hadron_tag
    f["ncfg"] = ncfg
    f["cfg_list"] = cfg_list
    f["pz_list"] = pz_list
    f["w_list"] = w_list
    f["tsep_list"] = tsep_list
    f["tau_list"] = tau_list
    f["hyp"] = hyp
    save_gevp_info(f, op_basis)
    f["phase_attached_to"] = phase_attached_to
print("saved", ratio_file)

#---- plot: saved in ratio_plots/{forward, sym_q..., sym_averaged_over_q}/pz{pz}
colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948", "#8c564b", "#17becf",
          "#7f7f7f", "#bcbd22", "#000000", "#c49c94", "#9edae5", "#ff9896", "#98df8a", "#c5b0d5", "#ffbb78", "#aec7e8"]
markers = ["o", "s", "^", "D", "v", "P", "X", "*", "<", ">", "p", "h", "H", "d", "8", "1", "2", "3", "4", "+"]
ratio_plot_dir = os.path.join(save_dir, "ratio_plots", ratio_subfolder)
for iw, w in enumerate(w_list):
    for ipz, pz in enumerate(pz_list):
        pz_dir = os.path.join(ratio_plot_dir, f"pz{pz}")
        os.makedirs(pz_dir, exist_ok=True)
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        for i, tsep in enumerate(plot_tsep_list):
            tau = np.arange(plot_tau_skip, tsep - plot_tau_skip + 1)
            m, e = jk(ratio_jk[plot_half][:, iw, ipz, tsep, tau].real)
            m = np.where(e < plot_err_max, m, np.nan)                           #NaN leaves a gap in the line instead of joining across the dropped points
            x = tau - tsep / 2 + 0.5 / len(plot_tsep_list) * (i - (len(plot_tsep_list) - 1) / 2)     #small shift so the error bars do not sit on top of each other   #small shift so the error bars do not sit on top of each other
            ax.errorbar(x, m, e, color=colors[i], marker=markers[i], ms=5, lw=0.8, elinewidth=1, capsize=2, label=f"tsep = {tsep}")
        ax.axhline(0, color="#898781", lw=0.8)
        ax.grid(color="#e1e0d9", lw=0.6)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_xlabel(r"$\tau - t_{\rm sep}/2$")
        ax.set_ylabel("Re R")
        ax.set_title(f"{hadron_tag}, {frame}, {ratio_q_tag}, {plot_half}, phase on {phase_attached_to}, hyp {hyp}, w = {w}, pz = {pz}, {ncfg} cfg", fontsize=10)
        ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)
        fig.tight_layout()
        fig.savefig(os.path.join(pz_dir, f"ratio_{frame}_{ratio_q_tag}_{gevp_tag}_phase{phase_attached_to}_{hadron_tag}_{plot_half}_hyp{hyp}_w{w}_pz{pz}.png"), dpi=150)
        plt.close(fig)
print("saved", len(w_list) * len(pz_list), "plots in", ratio_plot_dir)

GEVP choice basis1_t03_tref5
forward  -0.145(22) -0.300(32) -0.419(39) -0.549(62) -0.295(17) -0.323(39) -0.240(32)
backward -0.159(22) -0.296(33) -0.395(37) -0.436(59) -0.288(18) -0.319(40) -0.245(30)
fb       -0.152(21) -0.298(31) -0.407(35) -0.492(52) -0.291(17) -0.321(39) -0.242(30)
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/AI_assisted_matrix_element_analysis/basis1_t03_tref5/ratio_data/forward/ratio_forward_q0_0_0_basis1_t03_tref5_phase2pt_hyp8_ncfg250_pion.h5
saved 70 plots in /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/AI_assisted_matrix_element_analysis/basis1_t03_tref5/ratio_plots/forward


In [15]:
#======== sGEVP data: the summed 3pt of every operator pair, then R(t) and M_eff(t) with the GEVP vector of every t ========
#K_ab(T) = sum over tau = 1 .. T-1 of the 3pt of operators a, b, method 1 as in cell 4
#at every pz, jackknife sample and t > t0 of gevp_choice:  C(t) u = lambda C(t0) u,  u^dagger C(t0) u = 1,  u = the ground state at this t
#R(t) = u^dagger K(t) u / lambda(t) - u^dagger K(t0) u,   M_eff(t) = R(t+1) - R(t);  R(t0) = 0, and t < t0 is NaN
sgevp_q_average = True                          #sym only: True = average over the four q;  False = the q of cell 0 and -q in the GEVP
half_sign = {"forward": +1, "backward": -1}     #operator at t_src + tau (forward), t_src - tau (backward)

print("GEVP choice", gevp_tag)
assert phase_attached_to == "2pt", "this cell has the source phase on the 2pt, as in cell 4 with phase_attached_to = 2pt"

#---- which q go in, and the label of this data
if frame == "forward":
    sgevp_q_list = [(0, 0, 0)]                  #the q of the 3pt
    sgevp_C2_q_list = [(0, 0, 0)]               #the q of the 2pt in the GEVP
    sgevp_C2_type = "C3input"                   #at q = 0 this file is also C2pf and C2pi
    sgevp_q_tag = "q0_0_0"
    sgevp_subfolder = "forward"
elif sgevp_q_average:
    sgevp_q_list = q_list
    sgevp_C2_q_list = q_list
    sgevp_C2_type = "C2pf"
    sgevp_q_tag = "averaged_over_q"
    sgevp_subfolder = "sym_averaged_over_q"
else:
    sgevp_q_list = [q]
    sgevp_C2_q_list = [q, (-q[0], -q[1], -q[2])]
    sgevp_C2_type = "C2pf"
    sgevp_q_tag = f"q{q[0]}_{q[1]}_{q[2]}"
    sgevp_subfolder = f"sym_{sgevp_q_tag}"

#---- the 2pt matrix of the GEVP, for every jackknife sample
C2_cfg = {"forward": 0, "backward": 0}          #C2_cfg[half]: (cfg, op, op, pz, tsep)
for q_i in sgevp_C2_q_list:
    raw_forward, raw_backward = read_pt2(q_i, sgevp_C2_type)
    C2_cfg["forward"] = C2_cfg["forward"] + op_matrix(raw_forward.mean(axis=5), op_basis) / len(sgevp_C2_q_list)
    C2_cfg["backward"] = C2_cfg["backward"] + op_matrix(raw_backward.mean(axis=5), op_basis) / len(sgevp_C2_q_list)

#---- the summed 3pt of every operator pair, method 1, for every jackknife sample
K_jk = {"forward": 0, "backward": 0}            #K_jk[half]: (jk, op, op, w, pz, tsep), c_sign already in
for q_i in sgevp_q_list:
    print("summed 3pt at q =", q_i)
    C3input_file = pt2_template.format(frame=frame, qx=q_i[0], qy=q_i[1], qz=q_i[2], pt2_type="C3input", ncfg=ncfg, hadron=hadron_tag)
    with h5py.File(C3input_file, "r") as f:
        cfg_list = f["cfg_list"][()]
        pz_list = f["pz_list"][()]
        t_src_list = f["t_src_list"][()]        #(cfg, t_src): the slice of each t_src index, not sorted, shifts per cfg
    raw_forward, raw_backward = read_pt2(q_i, "C3input")
    raw_half = {"forward": raw_forward, "backward": raw_backward}

    #the gluon operator, as in cell 4: FF (cfg, munu, rhosig, hyp, w, t), munu order XY, YZ, XZ, XT, YT, ZT
    FF_file = os.path.join(pt2_dir, f"FF_hyp0-8_w0-9_q{q_i[0]}_{q_i[1]}_{q_i[2]}_ncfg800.h5")
    with h5py.File(FF_file, "r") as f:
        assert tuple(f["q"][()]) == q_i
        FF_cfg = f["cfg_list"][()]
        idx = np.searchsorted(FF_cfg, cfg_list)
        assert np.array_equal(FF_cfg[idx], cfg_list)
        ihyp = list(f["hyp_list"][()]).index(hyp)
        w_list = f["w_list"][()]
        TXTX = f["FF"][idx, 3, 3, ihyp]
        TYTY = f["FF"][idx, 4, 4, ihyp]
        XYXY = f["FF"][idx, 0, 0, ihyp]
    O = TXTX + TYTY - 2 * XYXY                  #(cfg, w, t)

    n_cfg, n_tsrc = t_src_list.shape
    i_cfg = np.arange(n_cfg)[:, None, None]
    for half in half_sign:
        Cij = op_matrix(raw_half[half], op_basis)                               #(cfg, op, op, t_src, pz, tsep)
        n_tsep = Cij.shape[-1]
        t_op = (t_src_list[:, :, None] + half_sign[half] * np.arange(n_tsep)) % Lt      #(cfg, t_src, tau)
        O_tau = O[i_cfg, :, t_op]                                               #(cfg, t_src, tau, w): numpy puts w last here
        O_sum = np.zeros_like(O_tau)                                            #(cfg, t_src, tsep, w)
        for T in range(n_tsep):
            O_sum[:, :, T] = O_tau[:, :, 1:T].sum(axis=2)                       #tau = 1 .. T-1
        connected = np.einsum("cabspT,csTw->cabwpT", Cij, O_sum, optimize=True) / n_tsrc                                                    #(cfg, op, op, w, pz, tsep)
        vacuum = np.einsum("cabpT,cTw->cabwpT", jackknife(Cij.mean(axis=3)), jackknife(O_sum.mean(axis=1)), optimize=True)                  #(jk, op, op, w, pz, tsep)
        K_jk[half] = K_jk[half] + c_sign * (jackknife(connected) - vacuum) / len(sgevp_q_list)
    del raw_forward, raw_backward, raw_half, Cij, connected, vacuum

#---- R(t), with the ground-state vector of every t and every jackknife sample
n_jk = K_jk["forward"].shape[0]
n_w = len(w_list)
n_pz = len(pz_list)
R_jk = {}                                       #R_jk[half]: (jk, w, pz, tsep)
for half in half_sign:
    C_jk = c_sign * jackknife(C2_cfg[half])                                     #(jk, op, op, pz, tsep)
    C_jk = 0.5 * (C_jk + C_jk.conj().swapaxes(1, 2))                            #hermitian part
    R_jk[half] = np.full((n_jk, n_w, n_pz, n_tsep), np.nan)
    for ipz, pz in enumerate(pz_list):
        use = [op_labels.index(name) for name in gevp_choice[pz]["basis"]]      #which of the six operators this pz uses
        t0_pz = gevp_choice[pz]["t0"]
        for i in range(n_jk):
            C_use = C_jk[i][np.ix_(use, use)][:, :, ipz]                        #(n_use, n_use, tsep)
            K_use = K_jk[half][i][np.ix_(use, use)][:, :, :, ipz]               #(n_use, n_use, w, tsep)
            val, vec = solve_gevp(C_use, t0_pz)                                 #val (state, tsep), vec (state, tsep, op);  state 0 is the ground state
            R_jk[half][i, :, ipz, t0_pz] = 0
            for T in range(t0_pz + 1, n_tsep):
                u = vec[0, T]                                                   #(n_use,)
                u_K_u = np.einsum("a,abw,b->w", u.conj(), K_use[:, :, :, T], u).real        #(w,)
                u_K0_u = np.einsum("a,abw,b->w", u.conj(), K_use[:, :, :, t0_pz], u).real   #(w,)
                R_jk[half][i, :, ipz, T] = u_K_u / val[0, T] - u_K0_u
R_jk["fb"] = 0.5 * (R_jk["forward"] + R_jk["backward"])

#---- M_eff(t) = R(t+1) - R(t)
Meff_jk = {}                                    #Meff_jk[half]: (jk, w, pz, t), NaN at the last t
for half in R_jk:
    Meff_jk[half] = np.full(R_jk[half].shape, np.nan)
    Meff_jk[half][:, :, :, :-1] = R_jk[half][:, :, :, 1:] - R_jk[half][:, :, :, :-1]

#check: M_eff at t0 + 2, w 0
m_text = []
for ipz, pz in enumerate(pz_list):
    m, e = jk(Meff_jk["fb"][:, 0, ipz, gevp_choice[pz]["t0"] + 2])
    m_text.append(f"{m:+.3f}({e * 1e3:.0f})")
print("M_eff(t0 + 2), w 0, fb:", " ".join(m_text))

#---- save
sgevp_file = os.path.join(save_dir, "sgevp_data", sgevp_subfolder, f"sgevp_{frame}_{sgevp_q_tag}_{gevp_tag}_phase{phase_attached_to}_hyp{hyp}_ncfg{ncfg}_{hadron_tag}.h5")
os.makedirs(os.path.dirname(sgevp_file), exist_ok=True)
with h5py.File(sgevp_file, "w") as f:
    for half in R_jk:
        f[f"R_{half}"] = R_jk[half]                 #(jk, w, pz, tsep);  half = forward, backward, fb
        f[f"Meff_{half}"] = Meff_jk[half]           #(jk, w, pz, t)
    f["q_list"] = sgevp_q_list                      #(n_q, 3): the q that went in
    f["q_tag"] = sgevp_q_tag
    f["subfolder"] = sgevp_subfolder
    f["frame"] = frame
    f["hadron_tag"] = hadron_tag
    f["ncfg"] = ncfg
    f["cfg_list"] = cfg_list
    f["pz_list"] = pz_list
    f["w_list"] = w_list
    f["tsep_list"] = np.arange(n_tsep)
    f["hyp"] = hyp
    save_gevp_info(f, op_basis)
    f["phase_attached_to"] = phase_attached_to
print("saved", sgevp_file)


GEVP choice basis1_t03_tref5
reading /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/2pt_coulomb_rho4p5z2p25-rho1p5_frac0p8-0_smear_GEVP_forward_q0_0_0_C3input_fb_tsep20_3072src_ncfg250_pion.h5
summed 3pt at q = (0, 0, 0)
reading /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/2pt_coulomb_rho4p5z2p25-rho1p5_frac0p8-0_smear_GEVP_forward_q0_0_0_C3input_fb_tsep20_3072src_ncfg250_pion.h5
M_eff(t0 + 2), w 0, fb: -0.161(26) -0.285(38) -0.398(41) -0.485(57) -0.287(23) -0.351(35) -0.254(26)
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/AI_assisted_matrix_element_analysis/basis1_t03_tref5/sgevp_data/forward/sgevp_forward_q0_0_0_basis1_t03_tref5_phase2pt_hyp8_ncfg250_pion.h5


In [16]:
#======== ratio fit, and its plots: every fit parameter is here ========
#the model: R = M00 + Ai (e^{-dE tau} + e^{-dE (tsep - tau)}) + Afi e^{-dE tsep},  fit_shared_gap and fit_cross_w_factor below set which w are fitted together
fit_ratio_file = ratio_file         #the file cell 5 just saved;  in a new kernel paste here the path that cell 5 printed, as a string
fit_half = "fb"                     #"forward", "backward" or "fb"
fit_window = {0: (6, 13, 3), 1: (5, 13, 2), 2: (4, 13, 2), 3: (4, 13, 2), 4: (4, 12, 2), 5: (4, 9, 2), 6: (4, 9, 2)}    #pz: (first tsep, last tsep, tau_skip);  tau = tau_skip .. tsep - tau_skip
fit_dE_prior = (0.5, 0.5)           #(center, width) of the gap at every pz and w;  the fit varies log(dE), so dE stays positive
fit_M00_prior = (-1.0, 10.0)        #(center, width);  the center must be negative, the fit varies log(-M00), so M00 stays negative
fit_Ai_prior = (0.0, 1e4)           #(center, width)
fit_Afi_prior = (0.0, 1e4)          #(center, width)
fit_correlated = True               #True = full covariance;  False = only its diagonal
fit_shared_gap = True              #True = one gap for all w at each pz;  False = each w has its own gap
fit_cross_w_factor = 1.0            #the off-w blocks of the covariance are multiplied by this:  0 = removed, 1 = the full all-w covariance
fit_svdcut = 1e-7
fit_maxit = 20000
fit_tol = 1e-8                      #stopping tolerance of the fit
fit_fitter = "scipy_least_squares"  #the minimizer
fit_label = "negM00"                      #extra text in the file name;  set it when you change a prior, svdcut, ..., or the old file is replaced

#---- read the ratio;  the labels of this fit come from the file
assert fit_half in ["forward", "backward", "fb"], f"fit_half = {fit_half}: use forward, backward or fb"
assert 0 <= fit_cross_w_factor <= 1, f"fit_cross_w_factor = {fit_cross_w_factor}: use a number from 0 to 1"
with h5py.File(fit_ratio_file, "r") as f:
    fit_data_jk = f[f"ratio_{fit_half}"][()]        #(jk, w, pz, tsep, tau), complex
    fit_w_list = f["w_list"][()]
    fit_pz_list = f["pz_list"][()]
    fit_q_tag = f["q_tag"].asstr()[()]
    fit_subfolder = f["subfolder"].asstr()[()]
    fit_frame = f["frame"].asstr()[()]
    fit_hadron = f["hadron_tag"].asstr()[()]
    fit_ncfg = f["ncfg"][()]
    fit_hyp = f["hyp"][()]
    fit_phase = f["phase_attached_to"].asstr()[()]
    fit_gevp_tag = f["gevp_tag"].asstr()[()]
print("ratio file", fit_ratio_file)
for pz in fit_pz_list:
    assert pz in fit_window, f"pz = {pz} is missing from fit_window"


#---- the model and the prior
if not gv.BufferDict.has_distribution("neglog"):       #a name for gvar, like its own "log":  the fit varies neglog(M00), and p["M00"] = -exp(neglog(M00))
    gv.BufferDict.add_distribution("neglog", lambda x: -gv.exp(x))


def ratio_model(x, p):
    #no denominator;  forward and sym: p_i and p_f have the same energies and overlaps, so one gap per fit of a w and Af = Ai
    #x["iw"] says which w of this fit each point belongs to
    tsep = x["tsep"]
    tau = x["tau"]
    iw = x["iw"]
    if fit_shared_gap:
        dE = p["dE"][0]
    else:
        dE = p["dE"][iw]
    return p["M00"][iw] + p["Ai"][iw] * (gv.exp(-dE * tau) + gv.exp(-dE * (tsep - tau))) + p["Afi"][iw] * gv.exp(-dE * tsep)


def ratio_prior(n_group):
    #one M00, Ai, Afi for each of the n_group values of w in this fit;  one gap for each of them, or one gap for all of them
    if fit_shared_gap:
        n_gap = 1
    else:
        n_gap = n_group
    prior = gv.BufferDict()
    prior["neglog(M00)"] = gv.log(-gv.gvar(np.full(n_group, float(fit_M00_prior[0])), np.full(n_group, float(fit_M00_prior[1]))))    #read it as fit.p["M00"], as with log(dE)
    prior["Ai"] = gv.gvar(np.full(n_group, float(fit_Ai_prior[0])), np.full(n_group, float(fit_Ai_prior[1])))
    prior["Afi"] = gv.gvar(np.full(n_group, float(fit_Afi_prior[0])), np.full(n_group, float(fit_Afi_prior[1])))
    prior["log(dE)"] = gv.log(gv.gvar(np.full(n_gap, float(fit_dE_prior[0])), np.full(n_gap, float(fit_dE_prior[1]))))    #read the gap as fit.p["dE"], fit.psdev["dE"] is not its error
    return prior


#---- first the jackknife mean, then every jackknife sample with the same covariance and prior, started from the fit of the mean
n_jk = fit_data_jk.shape[0]
n_w = len(fit_w_list)
n_pz = len(fit_pz_list)
fit_param_names = ["M00", "Ai", "Afi", "dE"]
param_jk = {}           #param_jk[name]: (jk, w, pz)
param_central = {}      #param_central[name]: (w, pz), the fit of the jackknife mean
param_lsqfit_err = {}   #param_lsqfit_err[name]: (w, pz), its lsqfit error, which sees the dE prior
for name in fit_param_names:
    param_jk[name] = np.zeros((n_jk, n_w, n_pz))
    param_central[name] = np.zeros((n_w, n_pz))
    param_lsqfit_err[name] = np.zeros((n_w, n_pz))
fit_chi2dof = np.zeros((n_w, n_pz))     #(w, pz)
fit_Q = np.zeros((n_w, n_pz))           #(w, pz)
fit_points = {}                         #fit_points[pz]: the tsep and tau of every point
fit_n_not_converged = 0                 #how many fits stopped at fit_maxit before converging;  lsqfit gives no warning for this
fit_n_total = 0                         #how many fits were done
for ipz, pz in enumerate(fit_pz_list):
    first, last, tau_skip = fit_window[pz]
    tsep_points = []
    tau_points = []
    for tsep in range(first, last + 1):
        for tau in range(tau_skip, tsep - tau_skip + 1):
            tsep_points.append(tsep)
            tau_points.append(tau)
    assert len(tsep_points) > 3, f"pz = {pz}: {len(tsep_points)} points for 3 free parameters, make the window larger"
    x = {"tsep": np.array(tsep_points), "tau": np.array(tau_points)}    #both (n_point,): the points of one w
    fit_points[pz] = x
    n_point = len(tsep_points)

    #which w are fitted together
    if fit_shared_gap or fit_cross_w_factor != 0:
        w_groups = [list(range(n_w))]       #all w in one fit
    else:
        w_groups = []
        for iw in range(n_w):
            w_groups.append([iw])           #each w in its own fit

    for group in w_groups:
        n_group = len(group)
        #the points of the w in this fit, one w after the other: all three (n_group * n_point,)
        x_group = {"tsep": np.tile(x["tsep"], n_group), "tau": np.tile(x["tau"], n_group), "iw": np.repeat(np.arange(n_group), n_point)}
        y_jk = fit_data_jk[:, group][:, :, ipz, x["tsep"], x["tau"]].real       #(jk, n_group, n_point): Re R
        y_jk = y_jk.reshape(n_jk, n_group * n_point)                            #(jk, n_group * n_point)
        assert not np.isnan(y_jk).any(), f"pz = {pz}: NaN in the ratio at the fit points, lower the last tsep in fit_window"
        cov = (n_jk - 1) * np.cov(y_jk, rowvar=False, bias=True)                #(n_group * n_point, n_group * n_point): jackknife covariance
        same_w = x_group["iw"][:, None] == x_group["iw"][None, :]               #True where both points have the same w
        cov = np.where(same_w, cov, fit_cross_w_factor * cov)                   #the off-w blocks are multiplied by the factor
        if not fit_correlated:
            cov = np.diag(np.diag(cov))
        if fit_cross_w_factor != 0 and n_group * n_point >= n_jk:
            print(f"warning: pz = {pz}: {n_group * n_point} points with {n_jk} samples, so the all-w covariance is not fully determined and the svdcut decides the small modes")

        gv.switch_gvar()    #the fits keep their covariances apart and gv.restore_gvar() below frees them:  without this, every rerun of this cell takes more memory and time
        fit = lsqfit.nonlinear_fit(data=(x_group, gv.gvar(y_jk.mean(axis=0), cov)), fcn=ratio_model, prior=ratio_prior(n_group), svdcut=fit_svdcut, maxit=fit_maxit, tol=fit_tol, fitter=fit_fitter)
        fit_n_total = fit_n_total + 1
        if fit.stopping_criterion == 0:
            fit_n_not_converged = fit_n_not_converged + 1
        dof = fit.dof       #data points + priors - parameters;  every parameter has a prior, so this equals the data points of this fit
        for k, iw in enumerate(group):
            fit_chi2dof[iw, ipz] = fit.chi2 / dof       #when all w are fitted together, every w shows the chi2/dof of that one fit
            fit_Q[iw, ipz] = lsqfit.gammaQ(dof / 2, fit.chi2 / 2)
            for name in fit_param_names:
                if name == "dE" and fit_shared_gap:
                    value = fit.p["dE"][0]
                else:
                    value = fit.p[name][k]
                param_central[name][iw, ipz] = value.mean
                param_lsqfit_err[name][iw, ipz] = value.sdev

        for i in range(n_jk):
            fit_i = lsqfit.nonlinear_fit(data=(x_group, gv.gvar(y_jk[i], cov)), fcn=ratio_model, prior=ratio_prior(n_group), p0=fit.pmean, svdcut=fit_svdcut, maxit=fit_maxit, tol=fit_tol, fitter=fit_fitter)
            fit_n_total = fit_n_total + 1
            if fit_i.stopping_criterion == 0:
                fit_n_not_converged = fit_n_not_converged + 1
            for k, iw in enumerate(group):
                for name in fit_param_names:
                    if name == "dE" and fit_shared_gap:
                        param_jk[name][i, iw, ipz] = fit_i.pmean["dE"][0]
                    else:
                        param_jk[name][i, iw, ipz] = fit_i.pmean[name][k]
        gv.restore_gvar()

#---- print the result: jk = jackknife mean and error, lsqfit = the fit of the mean, its error sees the dE prior
print(f"{fit_hadron}, {fit_frame}, {fit_q_tag}, {fit_half}, hyp {fit_hyp}, shared gap {fit_shared_gap}, cross-w factor {fit_cross_w_factor}")
print(" pz   w  points       M00 jk        dE jk   M00 lsqfit   chi2/dof      Q")
for ipz, pz in enumerate(fit_pz_list):
    for iw, w in enumerate(fit_w_list):
        mean, err = jk(param_jk["M00"][:, iw, ipz])
        M00_text = str(gv.gvar(mean, err))
        mean, err = jk(param_jk["dE"][:, iw, ipz])
        dE_text = str(gv.gvar(mean, err))
        M00_lsqfit_text = str(gv.gvar(param_central["M00"][iw, ipz], param_lsqfit_err["M00"][iw, ipz]))
        print(f"{pz:3d} {w:3d} {len(fit_points[pz]['tsep']):7d} {M00_text:>12s} {dE_text:>12s} {M00_lsqfit_text:>12s} {fit_chi2dof[iw, ipz]:10.2f} {fit_Q[iw, ipz]:6.3f}")
print(f"fits that stopped at fit_maxit before converging: {fit_n_not_converged} of {fit_n_total}")

#---- save: the file states every setting
fit_range_tag = "_".join(f"{fit_window[pz][0]}-{fit_window[pz][1]}s{fit_window[pz][2]}" for pz in fit_pz_list)     #first tsep - last tsep, tau_skip, one per pz
if fit_label == "":
    fit_label_text = ""
else:
    fit_label_text = "_" + fit_label
fit_w_text = ""
if fit_shared_gap:
    fit_w_text = fit_w_text + "_sharedgap"
if fit_cross_w_factor != 0:
    fit_w_text = fit_w_text + f"_xw{fit_cross_w_factor:g}"
fit_w_text = fit_w_text + f"_svd{fit_svdcut:g}"
fit_file = os.path.join(save_dir, "ratio_fit_data", fit_subfolder, f"ratio_fit_{fit_frame}_{fit_q_tag}_{fit_gevp_tag}_tsep{fit_range_tag}_phase{fit_phase}_ratio_{fit_half}_hyp{fit_hyp}_ncfg{fit_ncfg}{fit_w_text}{fit_label_text}_{fit_hadron}.h5")
if os.path.exists(fit_file):
    print("replacing", fit_file)
os.makedirs(os.path.dirname(fit_file), exist_ok=True)
with h5py.File(fit_file, "w") as f, h5py.File(fit_ratio_file, "r") as f_ratio:
    #results
    for name in fit_param_names:
        f[f"{name}_jk"] = param_jk[name]                    #(jk, w, pz)
        f[f"{name}_central"] = param_central[name]          #(w, pz)
        f[f"{name}_lsqfit_err"] = param_lsqfit_err[name]    #(w, pz)
    f["chi2dof"] = fit_chi2dof                              #(w, pz)
    f["Q"] = fit_Q                                          #(w, pz)
    f["n_not_converged"] = fit_n_not_converged
    #settings
    f["ratio_file"] = fit_ratio_file
    f["model"] = "ratio"
    f["fit_half"] = fit_half
    f["fit_window"] = [fit_window[pz] for pz in fit_pz_list]    #(pz, 3): first tsep, last tsep, tau_skip
    f["fit_range_tag"] = fit_range_tag
    f["dE_prior"] = fit_dE_prior                            #(center, width)
    f["M00_prior"] = fit_M00_prior                          #(center, width)
    f["Ai_prior"] = fit_Ai_prior                            #(center, width)
    f["Afi_prior"] = fit_Afi_prior                          #(center, width)
    f["correlated"] = fit_correlated
    f["shared_gap"] = fit_shared_gap
    f["cross_w_factor"] = fit_cross_w_factor
    f["svdcut"] = fit_svdcut
    f["maxit"] = fit_maxit
    f["tol"] = fit_tol
    f["fitter"] = fit_fitter
    f["fit_label"] = fit_label
    #what the data are, as stated in the ratio file
    for name in ["q_list", "q_tag", "subfolder", "frame", "hadron_tag", "ncfg", "cfg_list", "pz_list", "w_list", "hyp", "phase_attached_to",
                 "gevp_tag", "gevp_basis", "op_basis", "smear_names", "smear_list", "gamma_names", "t0", "t_ref"]:
        f_ratio.copy(name, f)
print("saved", fit_file)

#---- plot: saved in ratio_fit_plots/{forward, sym_q..., sym_averaged_over_q}/pz{pz}
#points: the fitted Re R;  colored bands: the fit curve of every jackknife sample, one per tsep;  gray band: M00
colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948", "#8c564b", "#17becf",
          "#7f7f7f", "#bcbd22", "#000000", "#c49c94", "#9edae5", "#ff9896", "#98df8a", "#c5b0d5", "#ffbb78", "#aec7e8"]
markers = ["o", "s", "^", "D", "v", "P", "X", "*", "<", ">", "p", "h", "H", "d", "8", "1", "2", "3", "4", "+"]
fit_plot_dir = os.path.join(save_dir, "ratio_fit_plots", fit_subfolder)
for ipz, pz in enumerate(fit_pz_list):
    pz_dir = os.path.join(fit_plot_dir, f"pz{pz}")
    os.makedirs(pz_dir, exist_ok=True)
    x = fit_points[pz]
    tsep_values = np.unique(x["tsep"])
    first, last, tau_skip = fit_window[pz]
    for iw, w in enumerate(fit_w_list):
        y_jk = fit_data_jk[:, iw, ipz, x["tsep"], x["tau"]].real        #(jk, n_point)
        p_jk = {}
        for name in fit_param_names:
            p_jk[name] = param_jk[name][None, :, iw, ipz, None]         #(1, jk, 1):  p_jk[name][0] is (jk, 1), so the model below gives (jk, n_tau)
        mean, err = jk(param_jk["M00"][:, iw, ipz])
        M00 = gv.gvar(mean, err)
        mean, err = jk(param_jk["dE"][:, iw, ipz])
        dE = gv.gvar(mean, err)
        M00_lsqfit = gv.gvar(param_central["M00"][iw, ipz], param_lsqfit_err["M00"][iw, ipz])
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        ax.axhspan(M00.mean - M00.sdev, M00.mean + M00.sdev, color="#e7e7e5", lw=0, zorder=0, label="M00")
        for i, tsep in enumerate(tsep_values):
            on_tsep = x["tsep"] == tsep
            tau = x["tau"][on_tsep]
            shift = 0.5 / len(tsep_values) * (i - (len(tsep_values) - 1) / 2)     #small shift so the error bars do not sit on top of each other    #small shift so the error bars do not sit on top of each other
            #0.25 past the last points on each side, else the tsep with one point has no width
            tau_band = np.linspace(tau[0] - 0.25, tau[-1] + 0.25, 40)
            band_mean, band_err = jk(ratio_model({"tsep": tsep, "tau": tau_band, "iw": 0}, p_jk))
            ax.fill_between(tau_band - tsep / 2 + shift, band_mean - band_err, band_mean + band_err, color=colors[i], alpha=0.25, lw=0, zorder=1)
            m, e = jk(y_jk[:, on_tsep])
            ax.errorbar(tau - tsep / 2 + shift, m, e, color=colors[i], marker=markers[i], ms=5, ls="none", elinewidth=1, capsize=2, zorder=2, label=f"tsep = {tsep}")
        ax.grid(color="#e1e0d9", lw=0.6)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_xlabel(r"$\tau - t_{\rm sep}/2$")
        ax.set_ylabel("Re R")
        ax.set_title(f"{fit_hadron}, {fit_frame}, {fit_q_tag}, {fit_half}, phase on {fit_phase}, hyp {fit_hyp}, w = {w}, pz = {pz}, {fit_ncfg} cfg{fit_w_text}\n"
                     f"M00 = {M00} (lsqfit {M00_lsqfit}), dE = {dE}, chi2/dof {fit_chi2dof[iw, ipz]:.2f}, Q {fit_Q[iw, ipz]:.3f}", fontsize=9)
        ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)
        fig.tight_layout()
        fig.savefig(os.path.join(pz_dir, f"ratio_fit_{fit_frame}_{fit_q_tag}_{fit_gevp_tag}_tsep{first}-{last}s{tau_skip}_phase{fit_phase}_{fit_hadron}_ratio_{fit_half}_hyp{fit_hyp}{fit_w_text}{fit_label_text}_w{w}_pz{pz}.png"), dpi=150)
        plt.close(fig)
print("saved", n_w * n_pz, "plots in", fit_plot_dir)

ratio file /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/AI_assisted_matrix_element_analysis/basis1_t03_tref5/ratio_data/forward/ratio_forward_q0_0_0_basis1_t03_tref5_phase2pt_hyp8_ncfg250_pion.h5


KeyboardInterrupt: 

In [ ]:
#======== sGEVP fit, and its plots: every fit parameter is here ========
#the model: M_eff(t) = M00 + B t e^{-dE t},  M00 and B for every w, one gap for all w at each pz
sg_file = sgevp_file                #the file of the sGEVP data cell;  in a new kernel paste here the path it printed, as a string
sg_half = "fb"                      #"forward", "backward" or "fb"
sg_window = {0: (4, 9), 1: (4, 9), 2: (4, 9), 3: (4, 8), 4: (4, 7), 5: (4, 7), 6: (4, 7)}    #pz: (first t, last t);  the first t has to be above the t0 of this pz
sg_correlated = True                #True = full covariance;  False = only its diagonal
sg_cross_w_factor = 1.0             #the off-w blocks of the covariance are multiplied by this:  0 = removed, 1 = the full all-w covariance
sg_dE_min = 0.25                    #lower bound of the gap, keep it at or above 1 / first t:  below that t e^{-dE t} does not fall across the window and can't be told from M00
sg_dE_prior = (0.25, 0.25)          #(center, width) of dE - sg_dE_min;  the fit varies log(dE - sg_dE_min), so dE stays above sg_dE_min;  this center puts the middle of the prior at dE = 0.5
sg_dE_start = [0.3, 0.5, 0.8, 1.2, 2.0]     #the fit of the mean starts from each of these gaps and keeps the lowest chi2, because it can stop in a local minimum
sg_M00_prior = (0.0, 100.0)         #(center, width)
sg_B_prior = (0.0, 100.0)           #(center, width)
sg_svdcut = 0.0                     #0 = no cut
sg_maxit = 20000
sg_tol = 1e-8                       #stopping tolerance of the fit
sg_fitter = "scipy_least_squares"   #the minimizer
sg_label = ""                       #extra text in the file name;  set it when you change a prior, svdcut, ..., or the old file is replaced

#---- read the data;  the labels of this fit come from the file
assert sg_half in ["forward", "backward", "fb"], f"sg_half = {sg_half}: use forward, backward or fb"
assert 0 <= sg_cross_w_factor <= 1, f"sg_cross_w_factor = {sg_cross_w_factor}: use a number from 0 to 1"
with h5py.File(sg_file, "r") as f:
    sg_data_jk = f[f"Meff_{sg_half}"][()]           #(jk, w, pz, t)
    sg_w_list = f["w_list"][()]
    sg_pz_list = f["pz_list"][()]
    sg_q_tag = f["q_tag"].asstr()[()]
    sg_subfolder = f["subfolder"].asstr()[()]
    sg_frame = f["frame"].asstr()[()]
    sg_hadron = f["hadron_tag"].asstr()[()]
    sg_ncfg = f["ncfg"][()]
    sg_hyp = f["hyp"][()]
    sg_phase = f["phase_attached_to"].asstr()[()]
    sg_gevp_tag = f["gevp_tag"].asstr()[()]
    sg_t0 = f["t0"][()]                             #(pz,)
print("sGEVP file", sg_file)
for ipz, pz in enumerate(sg_pz_list):
    assert pz in sg_window, f"pz = {pz} is missing from sg_window"
    assert sg_window[pz][0] > sg_t0[ipz], f"pz = {pz}: the first t of sg_window has to be above t0 = {sg_t0[ipz]}"
    assert sg_window[pz][1] - sg_window[pz][0] >= 2, f"pz = {pz}: use at least 3 t in sg_window"


#---- the model and the prior
def sgevp_model(x, p):
    #x["t"], x["iw"]: the t of every point and which w it belongs to, both (n_point,)
    dE = sg_dE_min + p["dE_above"][0]
    return p["M00"][x["iw"]] + p["B"][x["iw"]] * x["t"] * gv.exp(-dE * x["t"])


def sgevp_prior():
    prior = gv.BufferDict()
    prior["M00"] = gv.gvar(np.full(n_w, float(sg_M00_prior[0])), np.full(n_w, float(sg_M00_prior[1])))
    prior["B"] = gv.gvar(np.full(n_w, float(sg_B_prior[0])), np.full(n_w, float(sg_B_prior[1])))
    prior["log(dE_above)"] = gv.log(gv.gvar(np.full(1, float(sg_dE_prior[0])), np.full(1, float(sg_dE_prior[1]))))     #read it as fit.p["dE_above"]
    return prior


#---- first the jackknife mean from every start, then every jackknife sample with the same covariance and prior, started from the fit of the mean
n_jk = sg_data_jk.shape[0]
n_w = len(sg_w_list)
n_pz = len(sg_pz_list)
sg_param_names = ["M00", "B", "dE"]
sg_param_jk = {}                    #sg_param_jk[name]: (jk, w, pz)
sg_param_central = {}               #sg_param_central[name]: (w, pz), the fit of the jackknife mean
sg_param_lsqfit_err = {}            #sg_param_lsqfit_err[name]: (w, pz), its lsqfit error
for name in sg_param_names:
    sg_param_jk[name] = np.zeros((n_jk, n_w, n_pz))
    sg_param_central[name] = np.zeros((n_w, n_pz))
    sg_param_lsqfit_err[name] = np.zeros((n_w, n_pz))
sg_chi2dof = np.zeros(n_pz)         #(pz,): the one fit of all w
sg_Q = np.zeros(n_pz)               #(pz,)
sg_n_not_converged = 0              #how many fits stopped at sg_maxit before converging;  lsqfit gives no warning for this
sg_n_total = 0                      #how many fits were done
for ipz, pz in enumerate(sg_pz_list):
    first, last = sg_window[pz]
    t = np.arange(first, last + 1)                                              #(n_t,)
    n_t = len(t)
    x = {"t": np.tile(t, n_w), "iw": np.repeat(np.arange(n_w), n_t)}            #both (n_w * n_t,): the points of all w, one w after the other
    y_jk = sg_data_jk[:, :, ipz, t]                                             #(jk, n_w, n_t)
    assert not np.isnan(y_jk).any(), f"pz = {pz}: NaN in M_eff at the fit points, check sg_window"
    y_start = y_jk.mean(axis=0)                                                 #(n_w, n_t)
    y_jk = y_jk.reshape(n_jk, n_w * n_t)                                        #(jk, n_w * n_t)
    cov = (n_jk - 1) * np.cov(y_jk, rowvar=False, bias=True)                    #(n_w * n_t, n_w * n_t): jackknife covariance
    same_w = x["iw"][:, None] == x["iw"][None, :]                               #True where both points have the same w
    cov = np.where(same_w, cov, sg_cross_w_factor * cov)                        #the off-w blocks are multiplied by the factor
    if not sg_correlated:
        cov = np.diag(np.diag(cov))
    if sg_cross_w_factor != 0 and n_w * n_t >= n_jk:
        print(f"warning: pz = {pz}: {n_w * n_t} points with {n_jk} samples, so the all-w covariance is not fully determined")

    gv.switch_gvar()    #the fits keep their covariances apart and gv.restore_gvar() below frees them
    fit = None
    chi2_text = []
    for dE_start in sg_dE_start:
        p0 = {"M00": y_start[:, -1], "B": (y_start[:, 0] - y_start[:, -1]) / (first * np.exp(-dE_start * first)), "log(dE_above)": np.log([dE_start - sg_dE_min])}
        fit_s = lsqfit.nonlinear_fit(data=(x, gv.gvar(y_jk.mean(axis=0), cov)), fcn=sgevp_model, prior=sgevp_prior(), p0=p0, svdcut=sg_svdcut, maxit=sg_maxit, tol=sg_tol, fitter=sg_fitter)
        sg_n_total = sg_n_total + 1
        if fit_s.stopping_criterion == 0:
            sg_n_not_converged = sg_n_not_converged + 1
        chi2_text.append(f"{fit_s.chi2:.2f}")
        if fit is None or fit_s.chi2 < fit.chi2:
            fit = fit_s
    print(f"pz = {pz}: chi2 from the starts {sg_dE_start}: {' '.join(chi2_text)}")
    dof = fit.dof       #data points + priors - parameters;  every parameter has a prior, so this equals the data points of this fit
    sg_chi2dof[ipz] = fit.chi2 / dof
    sg_Q[ipz] = lsqfit.gammaQ(dof / 2, fit.chi2 / 2)
    dE_fit = sg_dE_min + fit.p["dE_above"][0]
    if dE_fit.mean < sg_dE_min + 0.01:
        print(f"warning: pz = {pz}: the gap sits at its lower bound {sg_dE_min}, M00 of this pz is not determined")
    for iw in range(n_w):
        sg_param_central["M00"][iw, ipz] = fit.p["M00"][iw].mean
        sg_param_lsqfit_err["M00"][iw, ipz] = fit.p["M00"][iw].sdev
        sg_param_central["B"][iw, ipz] = fit.p["B"][iw].mean
        sg_param_lsqfit_err["B"][iw, ipz] = fit.p["B"][iw].sdev
        sg_param_central["dE"][iw, ipz] = dE_fit.mean
        sg_param_lsqfit_err["dE"][iw, ipz] = dE_fit.sdev

    for i in range(n_jk):
        fit_i = lsqfit.nonlinear_fit(data=(x, gv.gvar(y_jk[i], cov)), fcn=sgevp_model, prior=sgevp_prior(), p0=fit.pmean, svdcut=sg_svdcut, maxit=sg_maxit, tol=sg_tol, fitter=sg_fitter)
        sg_n_total = sg_n_total + 1
        if fit_i.stopping_criterion == 0:
            sg_n_not_converged = sg_n_not_converged + 1
        sg_param_jk["M00"][i, :, ipz] = fit_i.pmean["M00"]
        sg_param_jk["B"][i, :, ipz] = fit_i.pmean["B"]
        sg_param_jk["dE"][i, :, ipz] = sg_dE_min + fit_i.pmean["dE_above"][0]
    gv.restore_gvar()

#---- print the result: jk = jackknife mean and error, lsqfit = the fit of the mean
print(f"{sg_hadron}, {sg_frame}, {sg_q_tag}, {sg_half}, hyp {sg_hyp}, cross-w factor {sg_cross_w_factor}")
print(" pz   w    t       M00 jk        dE jk   M00 lsqfit   chi2/dof      Q")
for ipz, pz in enumerate(sg_pz_list):
    for iw, w in enumerate(sg_w_list):
        mean, err = jk(sg_param_jk["M00"][:, iw, ipz])
        M00_text = str(gv.gvar(mean, err))
        mean, err = jk(sg_param_jk["dE"][:, iw, ipz])
        dE_text = str(gv.gvar(mean, err))
        M00_lsqfit_text = str(gv.gvar(sg_param_central["M00"][iw, ipz], sg_param_lsqfit_err["M00"][iw, ipz]))
        print(f"{pz:3d} {w:3d} {sg_window[pz][0]:2d}-{sg_window[pz][1]:<2d} {M00_text:>12s} {dE_text:>12s} {M00_lsqfit_text:>12s} {sg_chi2dof[ipz]:10.2f} {sg_Q[ipz]:6.3f}")
print(f"fits that stopped at sg_maxit before converging: {sg_n_not_converged} of {sg_n_total}")

#---- save: the file states every setting
sg_range_tag = "_".join(f"{sg_window[pz][0]}-{sg_window[pz][1]}" for pz in sg_pz_list)     #first t - last t, one per pz
if sg_label == "":
    sg_label_text = ""
else:
    sg_label_text = "_" + sg_label
sg_xw_text = f"_xw{sg_cross_w_factor:g}_svd{sg_svdcut:g}"
sg_fit_file = os.path.join(save_dir, "sgevp_fit_data", sg_subfolder, f"sgevp_fit_{sg_frame}_{sg_q_tag}_{sg_gevp_tag}_t{sg_range_tag}_phase{sg_phase}_{sg_half}_hyp{sg_hyp}_ncfg{sg_ncfg}{sg_xw_text}{sg_label_text}_{sg_hadron}.h5")
if os.path.exists(sg_fit_file):
    print("replacing", sg_fit_file)
os.makedirs(os.path.dirname(sg_fit_file), exist_ok=True)
with h5py.File(sg_fit_file, "w") as f, h5py.File(sg_file, "r") as f_in:
    #results
    for name in sg_param_names:
        f[f"{name}_jk"] = sg_param_jk[name]                     #(jk, w, pz)
        f[f"{name}_central"] = sg_param_central[name]           #(w, pz)
        f[f"{name}_lsqfit_err"] = sg_param_lsqfit_err[name]     #(w, pz)
    f["chi2dof"] = sg_chi2dof                                   #(pz,)
    f["Q"] = sg_Q                                               #(pz,)
    f["n_not_converged"] = sg_n_not_converged
    #settings
    f["sgevp_file"] = sg_file
    f["model"] = "sgevp"
    f["fit_half"] = sg_half
    f["fit_window"] = [sg_window[pz] for pz in sg_pz_list]      #(pz, 2): first t, last t
    f["fit_range_tag"] = sg_range_tag
    f["correlated"] = sg_correlated
    f["cross_w_factor"] = sg_cross_w_factor
    f["dE_min"] = sg_dE_min
    f["dE_prior"] = sg_dE_prior                                 #(center, width) of dE - dE_min
    f["dE_start"] = sg_dE_start
    f["M00_prior"] = sg_M00_prior                               #(center, width)
    f["B_prior"] = sg_B_prior                                   #(center, width)
    f["svdcut"] = sg_svdcut
    f["maxit"] = sg_maxit
    f["tol"] = sg_tol
    f["fitter"] = sg_fitter
    f["fit_label"] = sg_label
    #what the data are, as stated in the sGEVP file
    for name in ["q_list", "q_tag", "subfolder", "frame", "hadron_tag", "ncfg", "cfg_list", "pz_list", "w_list", "hyp", "phase_attached_to",
                 "gevp_tag", "gevp_basis", "op_basis", "smear_names", "smear_list", "gamma_names", "t0", "t_ref"]:
        f_in.copy(name, f)
print("saved", sg_fit_file)

#---- plot: saved in sgevp_fit_plots/{forward, sym_q..., sym_averaged_over_q}/pz{pz}
#points: M_eff, filled inside the fit window, open outside;  blue band: the fit curve of every jackknife sample;  gray band: M00
sg_plot_t = np.arange(2, 13)
sg_plot_err_max = 0.5               #leave out points with a larger error, so noisy t do not set the y range
sg_color = "#2a78d6"
sg_plot_dir = os.path.join(save_dir, "sgevp_fit_plots", sg_subfolder)
for ipz, pz in enumerate(sg_pz_list):
    pz_dir = os.path.join(sg_plot_dir, f"pz{pz}")
    os.makedirs(pz_dir, exist_ok=True)
    first, last = sg_window[pz]
    t_band = np.linspace(first, last, 40)                       #(n_band,)
    for iw, w in enumerate(sg_w_list):
        m, e = jk(sg_data_jk[:, iw, ipz, sg_plot_t])
        m = np.where(e < sg_plot_err_max, m, np.nan)
        in_fit = (sg_plot_t >= first) & (sg_plot_t <= last)
        M00_i = sg_param_jk["M00"][:, iw, ipz, None]            #(jk, 1)
        B_i = sg_param_jk["B"][:, iw, ipz, None]                #(jk, 1)
        dE_i = sg_param_jk["dE"][:, iw, ipz, None]              #(jk, 1)
        band_mean, band_err = jk(M00_i + B_i * t_band * np.exp(-dE_i * t_band))
        mean, err = jk(sg_param_jk["M00"][:, iw, ipz])
        M00 = gv.gvar(mean, err)
        mean, err = jk(sg_param_jk["dE"][:, iw, ipz])
        dE = gv.gvar(mean, err)
        M00_lsqfit = gv.gvar(sg_param_central["M00"][iw, ipz], sg_param_lsqfit_err["M00"][iw, ipz])
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        ax.axhspan(M00.mean - M00.sdev, M00.mean + M00.sdev, color="#e7e7e5", lw=0, zorder=0, label="M00")
        ax.fill_between(t_band, band_mean - band_err, band_mean + band_err, color=sg_color, alpha=0.25, lw=0, zorder=1, label="fit")
        ax.errorbar(sg_plot_t[in_fit], m[in_fit], e[in_fit], color=sg_color, marker="o", ms=5, ls="none", elinewidth=1, capsize=2, zorder=2, label="in the fit")
        ax.errorbar(sg_plot_t[~in_fit], m[~in_fit], e[~in_fit], color=sg_color, marker="o", mfc="white", ms=5, ls="none", elinewidth=1, capsize=2, zorder=2, label="not in the fit")
        ax.axhline(0, color="#898781", lw=0.8)
        ax.grid(color="#e1e0d9", lw=0.6)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_xlabel("t")
        ax.set_ylabel(r"$M_{\rm eff}(t)$")
        ax.set_title(f"{sg_hadron}, {sg_frame}, {sg_q_tag}, {sg_half}, phase on {sg_phase}, hyp {sg_hyp}, w = {w}, pz = {pz}, {sg_ncfg} cfg{sg_xw_text}\n"
                     f"M00 = {M00} (lsqfit {M00_lsqfit}), dE = {dE}, chi2/dof {sg_chi2dof[ipz]:.2f}, Q {sg_Q[ipz]:.3f}", fontsize=9)
        ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)
        fig.tight_layout()
        fig.savefig(os.path.join(pz_dir, f"sgevp_fit_{sg_frame}_{sg_q_tag}_{sg_gevp_tag}_t{first}-{last}_phase{sg_phase}_{sg_hadron}_{sg_half}_hyp{sg_hyp}{sg_xw_text}{sg_label_text}_w{w}_pz{pz}.png"), dpi=150)
        plt.close(fig)
print("saved", n_w * n_pz, "plots in", sg_plot_dir)


In [ ]:
#======== single and double ratios: the fits of this gevp_tag, with the model of itd_model;  only the first cell has to be run before this one ========
import gvar as gv

#---- what to use
itd_model = "ratio"                 #"ratio" = the fit of cell 6;  "sgevp" = the fit of the sGEVP fit cell
#only when this folder has several fits: text in the name of the one to use at each pz, e.g. "sharedgap_xw1" or "t3-12n3";  "" = there is only one
itd_choice = {}
for pz in gevp_choice:
    itd_choice[pz] = {"ratio_fit": "", "sgevp_fit": "", "2pt_fit": ""}
itd_ref_choice = {"ratio_fit": "", "sgevp_fit": "", "2pt_fit": ""}    #only for sym with itd_ref = "p000": the fit of p = (0,0,0) in the forward run of this folder
itd_label = itd_model               #goes in the names of the saved file and plot:  change it when you change the fits above, or the old file is replaced
itd_average_over_q = True           #sym only: True = the fits of the q average, False = the fits at the q of the first cell
itd_ref = "pz0"                     #sym only, the reference at nu = 0: "pz0" = the P_z = 0 entry of this run (the same t), "p000" = p = (0,0,0) of the forward run (t = 0);  forward: always p = (0,0,0)
itd_w_ref = 0                       #double ratio: every matrix element is divided by its value at this w
itd_plot_err_max = 0.5              #leave out points with a larger error, so noisy points do not set the y range

itd_root = save_dir
assert itd_model in ["ratio", "sgevp"], f"itd_model = {itd_model}: use ratio or sgevp"
for choice in list(itd_choice.values()) + [itd_ref_choice]:
    choice["gevp_tag"] = gevp_tag
itd_q_tag = "averaged_over_q" if itd_average_over_q and frame == "sym" else f"q{q[0]}_{q[1]}_{q[2]}"
itd_subfolder = "forward" if frame == "forward" else f"sym_{itd_q_tag}"
assert itd_ref in ["pz0", "p000"], f"itd_ref = {itd_ref}: use pz0 or p000"
itd_ref_used = "p000" if frame == "forward" else itd_ref


#---- find and read the fits
def find_fit(kind, fit_frame, q_tag_i, subfolder_i, choice):
    #the one saved fit with the GEVP tag of this choice;  kind = "ratio_fit" or "2pt_fit"
    folder = os.path.join(itd_root, f"{kind}_data", subfolder_i)
    files = glob.glob(os.path.join(folder, f"{kind}_{fit_frame}_{q_tag_i}_{choice['gevp_tag']}_*_{hadron_tag}.h5"))
    files = [file for file in files if choice[kind] in os.path.basename(file)]
    available = "\n".join(sorted(os.listdir(folder))) if os.path.isdir(folder) else f"no folder {folder}"
    assert len(files) == 1, f"{len(files)} {kind} files for tag {choice['gevp_tag']}, {hadron_tag}, {subfolder_i};  if there are several, put text from the name of the one to use in \"{kind}\" of this entry of itd_choice.  In the folder:\n{available}"
    return files[0]


def read_fits(fit_frame, q_tag_i, subfolder_i, choice):
    #M00 (jk, w, pz) of the ratio fit and the central E0 (pz,) of the 2pt fit with the same GEVP tag
    fits = {"ratio_fit_file": find_fit(itd_model + "_fit", fit_frame, q_tag_i, subfolder_i, choice), "pt2_fit_file": find_fit("2pt_fit", fit_frame, q_tag_i, subfolder_i, choice)}
    with h5py.File(fits["ratio_fit_file"], "r") as f, h5py.File(fits["pt2_fit_file"], "r") as g:
        for fit in [f, g]:
            assert fit["gevp_tag"].asstr()[()] == choice["gevp_tag"], f"{fit.filename}: the file states another GEVP tag"
        assert np.array_equal(f["cfg_list"][()], g["cfg_list"][()]) and np.array_equal(f["pz_list"][()], g["pz_list"][()]), "the ratio fit and the 2pt fit have other cfgs or pz"
        assert f["fit_half"][()] == g["fit_half"][()], "the ratio fit and the 2pt fit use another half"
        fits["M00_jk"] = f["M00_jk"][()]
        fits["E_central"] = g["E_central"][:, 0]
        fits["w_list"], fits["pz_list"], fits["cfg_list"] = f["w_list"][()], f["pz_list"][()], f["cfg_list"][()]
        fits["ratio_fit_range"], fits["pt2_fit_range"] = f["fit_range_tag"].asstr()[()], g["fit_range_tag"].asstr()[()]
        fits["Ls"] = g["Ls"][()]
        fits["hyp"], fits["phase"], fits["fit_half"] = f["hyp"][()], f["phase_attached_to"].asstr()[()], f["fit_half"].asstr()[()]
    return fits


run_fits = {}       #run_fits[pz]: the fits this pz uses
for pz in itd_choice:
    run_fits[pz] = read_fits(frame, itd_q_tag, itd_subfolder, itd_choice[pz])
first_fits = run_fits[min(itd_choice)]
w_values, pz_values = first_fits["w_list"], first_fits["pz_list"]
M_jk = np.zeros(first_fits["M00_jk"].shape)     #(jk, w, pz)
E_central = np.zeros(len(pz_values))            #(pz,)
for ipz, pz in enumerate(pz_values):
    assert pz in itd_choice, f"pz = {pz} is missing from itd_choice"
    for name in ["cfg_list", "pz_list", "w_list", "hyp", "phase", "fit_half"]:
        assert np.array_equal(run_fits[pz][name], first_fits[name]), f"pz = {pz}: the fits have another {name} than pz = {min(itd_choice)}"
    M_jk[:, :, ipz] = run_fits[pz]["M00_jk"][:, :, ipz]
    E_central[ipz] = run_fits[pz]["E_central"][ipz]
if itd_ref_used == "p000" and frame == "sym":
    ref_fits = read_fits("forward", "q0_0_0", "forward", itd_ref_choice)
    for name in ["cfg_list", "w_list", "hyp", "phase"]:
        assert np.array_equal(ref_fits[name], first_fits[name]), f"the forward run has another {name}"
else:
    ref_fits = run_fits[0]              #forward: pz 0 is p = (0,0,0);  sym with pz0: pz 0 is the P_z = 0 entry
ipz_ref = list(ref_fits["pz_list"]).index(0)
M_ref_jk = ref_fits["M00_jk"][:, :, ipz_ref, None]      #(jk, w, 1)
E_ref = ref_fits["E_central"][ipz_ref]


#---- the ratios, sample by sample;  E_ref / E is the central value for every sample
iw_ref = list(w_values).index(itd_w_ref)
nu = 2 * np.pi / first_fits["Ls"] * np.outer(w_values, pz_values)                                         #(w, pz)
single_jk = M_jk / M_ref_jk * (E_ref / E_central)                                                       #(jk, w, pz)
double_jk = (M_jk / M_jk[:, iw_ref:iw_ref + 1]) / (M_ref_jk / M_ref_jk[:, iw_ref:iw_ref + 1])           #(jk, w, pz)
single_mean, single_err = jk(single_jk)
double_mean, double_err = jk(double_jk)
print(f"{hadron_tag}, {frame}, {itd_q_tag}, {itd_label}, reference {itd_ref_used}, E_ref = {E_ref:.4f}")
for pz in pz_values:
    print(f"pz {pz} ratio fit:", os.path.basename(run_fits[pz]["ratio_fit_file"]))
    print(f"pz {pz} 2pt fit:  ", os.path.basename(run_fits[pz]["pt2_fit_file"]))
print(f"{'pz':>3s} {'E':>8s} {'E_ref/E':>8s} {f'S(w={w_values[0]})':>11s} {f'S(w={w_values[4]})':>11s} {f'S(w={w_values[8]})':>11s} {f'D(w={w_values[4]})':>11s} {f'D(w={w_values[8]})':>11s}")
for ipz, pz in enumerate(pz_values):
    print(f"{pz:3d} {E_central[ipz]:8.4f} {E_ref / E_central[ipz]:8.4f} " + " ".join(f"{str(gv.gvar(a, b)):>11s}" for a, b in [(single_mean[0, ipz], single_err[0, ipz]), (single_mean[4, ipz], single_err[4, ipz]), (single_mean[8, ipz], single_err[8, ipz]), (double_mean[4, ipz], double_err[4, ipz]), (double_mean[8, ipz], double_err[8, ipz])]))


#---- save
itd_name = f"itd_{frame}_{itd_q_tag}_{itd_label}_phase{first_fits['phase']}_{first_fits['fit_half']}_hyp{first_fits['hyp']}_ref{itd_ref_used}_wref{itd_w_ref}_{hadron_tag}"
itd_file = os.path.join(itd_root, "itd_data", itd_subfolder, itd_name + ".h5")
if os.path.exists(itd_file):
    print("replacing", itd_file)
os.makedirs(os.path.dirname(itd_file), exist_ok=True)
with h5py.File(itd_file, "w") as f:
    f["single_ratio_jk"] = single_jk    #(jk, w, pz): M(p, w) / M_ref(w) * E_ref / E
    f["double_ratio_jk"] = double_jk    #(jk, w, pz): [M(p, w) / M(p, w_ref)] / [M_ref(w) / M_ref(w_ref)]
    f["nu"] = nu                        #(w, pz): 2 pi / Ls * pz * w
    f["E"] = E_central                  #(pz,): the central E0 of the 2pt fit
    f["E_ref"] = E_ref
    f["w_ref"] = itd_w_ref
    f["reference"] = itd_ref_used
    f["label"] = itd_label
    f["ref_ratio_fit_file"] = ref_fits["ratio_fit_file"]
    f["ref_pt2_fit_file"] = ref_fits["pt2_fit_file"]
    with h5py.File(first_fits["ratio_fit_file"], "r") as fit_in:
        for name in ["w_list", "pz_list", "cfg_list", "q_list", "hyp", "fit_half", "phase_attached_to", "smear_names", "smear_list", "gamma_names"]:
            fit_in.copy(name, f)        #the same at every pz
    for pz in pz_values:
        #what the fits of this pz used:  f["choice/pz3/gevp_tag"] and so on
        f[f"choice/pz{pz}/ratio_fit_file"] = run_fits[pz]["ratio_fit_file"]
        f[f"choice/pz{pz}/pt2_fit_file"] = run_fits[pz]["pt2_fit_file"]
        f[f"choice/pz{pz}/pt2_fit_range_tag"] = run_fits[pz]["pt2_fit_range"]
        with h5py.File(run_fits[pz]["ratio_fit_file"], "r") as fit_in:
            for name in ["model", "fit_window", "fit_range_tag", "gevp_tag", "gevp_basis", "op_basis", "t0", "t_ref", "shared_gap", "cross_w_factor"]:
                if name in fit_in:      #shared_gap and cross_w_factor are not in older fits
                    fit_in.copy(name, f, name=f"choice/pz{pz}/{name}")
print("saved", itd_file)


#---- plot: the single and the double ratio versus nu, one line per pz
itd_colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]
itd_markers = ["o", "s", "^", "D", "v", "P", "X"]
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
for ax, mean, err, title in [(axes[0], single_mean, single_err, "single ratio:  M(p, w) / M_ref(w) x E_ref / E"),
                             (axes[1], double_mean, double_err, f"double ratio:  [M(p, w) / M(p, {itd_w_ref})] / [M_ref(w) / M_ref({itd_w_ref})]")]:
    mean = np.where(err < itd_plot_err_max, mean, np.nan)
    for ipz, pz in enumerate(pz_values):
        ax.errorbar(nu[:, ipz], mean[:, ipz], err[:, ipz], color=itd_colors[ipz], marker=itd_markers[ipz], ms=5, lw=0.8, elinewidth=1, capsize=2, label=f"pz = {pz}")
    ax.axhline(1, color="#52514e", lw=0.8, ls=":")
    ax.grid(color="#e1e0d9", lw=0.6)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel(r"$\nu = P_z z$")
    ax.set_ylabel("Re")
    ax.set_title(title, fontsize=10)
axes[1].legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)
fig.suptitle(f"{hadron_tag}, {frame}, {itd_q_tag}, reference {itd_ref_used}, {len(first_fits['cfg_list'])} cfg\n{itd_label}", fontsize=10)
fig.tight_layout()
itd_plot_file = os.path.join(itd_root, "itd_plots", itd_subfolder, itd_name + ".png")
os.makedirs(os.path.dirname(itd_plot_file), exist_ok=True)
fig.savefig(itd_plot_file, dpi=150)
plt.show()
print("saved", itd_plot_file)